# TA-HGMAE: Temporal-Aware Heterogeneous Graph Masked Autoencoder
## Procurement Fraud Detection Prototype

**Architecture:** HGT encoder + temporal feature engineering + masked autoencoding + dual loss

**Run cells sequentially.**

In [3]:
import torch

def format_pytorch_version(version):
  return version.split('+')[0]

torch_version = format_pytorch_version(torch.__version__)
CUDA_version = torch.version.cuda

!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch_version}+{CUDA_version}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch_version}+{CUDA_version}.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.10.0+None.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677289 sha256=294ad8cfe457df3b48a1fb671c43a6ab1bdc7930968de2aad70a0c4e30bfaafa
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
Successfully built torch-scatter
Looking in links: https://data.pyg.org/whl/torch-2.10.0+None.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=1261475 sha256=78f9d9b16c2e44150d49784b81e0331918aeb27d7bf57685b3e861b1d1a3a349
  Stored in directory: /root/.cache/pip/wheels/71/fa/21/bd1d78ce1629aec4ecc924a63b82f6949dda484b6321eac6f2
Successfully built torc

In [ ]:
import os, json, gzip, requests

os.makedirs("data/raw", exist_ok=True)

# URL for the 2022 Colombian procurement data
url = "https://data.open-contracting.org/en/publication/61/download?name=2022.jsonl.gz"
out_path = "data/raw/colombia_procurement_sample.jsonl"

MAX_RECORDS = 50000  # Number of records to download and process

print(f"Downloading and processing up to {MAX_RECORDS} records from {url}...")

try:
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status() # Raise an exception for HTTP errors
        gz = gzip.GzipFile(fileobj=r.raw)

        with open(out_path, "wt", encoding="utf-8") as out_file:
            for i, line in enumerate(gz):
                if i >= MAX_RECORDS:
                    break
                out_file.write(line.decode("utf-8"))

    print(f"Successfully saved {min(i+1, MAX_RECORDS)} records to {out_path}")
except requests.exceptions.RequestException as e:
    print(f"Error downloading data: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from collections import defaultdict
from typing import Dict, List, Tuple
from torch_geometric.data import HeteroData
import gzip 
import os 

# PART 1: OCD DATA LOADER

class OCDDataLoader:
    """
    Loads and parses Colombian Open Contracting Data (OCDS format).
    Handles the specific JSON structure from Colombia's SECOPI platform.
    """

    def __init__(self, filepath: str):
        """
        Args:
            filepath: Path to JSONL file (one JSON object per line)
        """
        self.filepath = filepath
        self.records = []

    def load_data(self, max_records: int = None) -> List[Dict]:
        """
        Load OCD data from JSONL file.

        Args:
            max_records: Maximum number of records to load (None = all)

        Returns:
            List of parsed records
        """
        print(f"Loading OCD data from {self.filepath}...")

        try:
            # Try opening as gzipped file first (using 'rt' for read text mode)
            with gzip.open(self.filepath, 'rt', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    if max_records and i >= max_records:
                        break

                    try:
                        record = json.loads(line.strip())
                        self.records.append(record)
                    except json.JSONDecodeError as e:
                        print(f"Warning: Skipping malformed record at line {i+1}: {e}")
                        continue
        except gzip.BadGzipFile:
            # If it's not a gzip file, try opening as a regular text file
            print(f"File {self.filepath} is not a gzip file, trying as plain text...")
            with open(self.filepath, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    if max_records and i >= max_records:
                        break

                    try:
                        record = json.loads(line.strip())
                        self.records.append(record)
                    except json.JSONDecodeError as e:
                        print(f"Warning: Skipping malformed record at line {i+1}: {e}")
                        continue
        except Exception as e:
            print(f"An unexpected error occurred while loading data: {e}")
            raise

        print(f"Loaded {len(self.records)} records")
        return self.records

    def parse_record(self, record: Dict) -> Dict:
        """
        Parse a single OCD record into structured format.

        Extracts:
        - Tender information
        - Buyer information
        - Items/classifications
        - Dates and amounts
        """
        parsed = {
            'tender_id': record.get('tender', {}).get('id'),
            'ocid': record.get('ocid'),
            'buyer_id': record.get('buyer', {}).get('id'),
            'buyer_name': record.get('buyer', {}).get('name'),
            'tender_title': record.get('tender', {}).get('title'),
            'tender_description': record.get('tender', {}).get('description'),
            'tender_value': record.get('tender', {}).get('value', {}).get('amount', 0),
            'currency': record.get('tender', {}).get('value', {}).get('currency', 'COP'),
            'tender_status': record.get('tender', {}).get('status'),
            'procurement_method': record.get('tender', {}).get('procurementMethod'),
            'procurement_method_details': record.get('tender', {}).get('procurementMethodDetails'),
            'start_date': self._parse_date(record.get('tender', {}).get('tenderPeriod', {}).get('startDate')),
            'end_date': self._parse_date(record.get('tender', {}).get('tenderPeriod', {}).get('endDate')),
            'region': self._extract_region(record),
            'locality': self._extract_locality(record),
            'items': self._extract_items(record),
        }

        return parsed

    def _parse_date(self, date_str: str) -> datetime:
        """Parse ISO date string to datetime."""
        if not date_str:
            return None
        try:
            return datetime.fromisoformat(date_str.replace('Z', '+00:00'))
        except:
            return None

    def _extract_region(self, record: Dict) -> str:
        """Extract region from parties information."""
        parties = record.get('parties', [])
        if parties and len(parties) > 0:
            return parties[0].get('address', {}).get('region', '')
        return ''

    def _extract_locality(self, record: Dict) -> str:
        """Extract locality (municipality) from parties information."""
        parties = record.get('parties', [])
        if parties and len(parties) > 0:
            return parties[0].get('address', {}).get('locality', '')
        return ''

    def _extract_items(self, record: Dict) -> List[Dict]:
        """Extract item classifications from tender."""
        items = record.get('tender', {}).get('items', [])
        return [
            {
                'id': item.get('id'),
                'description': item.get('description'),
                'classification_id': item.get('classification', {}).get('id'),
                'classification_desc': item.get('classification', {}).get('description'),
            }
            for item in items
        ]

    def to_dataframe(self) -> pd.DataFrame:
        """
        Convert loaded records to pandas DataFrame for analysis.
        """
        parsed_records = [self.parse_record(r) for r in self.records]
        df = pd.DataFrame(parsed_records)
        return df



# PART 2: HETEROGENEOUS GRAPH BUILDER

class ProcurementGraphBuilder:
    """
    Builds a heterogeneous temporal graph from OCD data.

    Graph Schema:
    - Node Types: Buyer, Tender
    - Edge Types: (Buyer, issues, Tender)

    Note: This is a simplified schema. Full implementation would include:
    - Supplier nodes (from awards data)
    - Contract nodes
    - Additional edge types
    """

    def __init__(self):
        self.buyers = {}
        self.tenders = {}
        self.edges = defaultdict(list)

    def build_graph(self, records: List[Dict]) -> HeteroData:
        """
        Build heterogeneous graph from parsed OCD records.

        Args:
            records: List of OCD records (raw JSON)

        Returns:
            HeteroData object for PyTorch Geometric
        """
        print("Building heterogeneous graph...")

        # Step 1: Extract and index nodes
        self._extract_nodes(records)

        # Step 2: Create edges
        self._create_edges(records)

        # Step 3: Convert to PyTorch Geometric format
        graph = self._to_hetero_data()

        print(f"Graph built successfully:")
        print(f"  - Buyers: {len(self.buyers)}")
        print(f"  - Tenders: {len(self.tenders)}")
        print(f"  - Edges: {sum(len(e) for e in self.edges.values())}")

        return graph

    def _extract_nodes(self, records: List[Dict]):
        """Extract and index unique buyers and tenders."""

        for record in records:
            # Extract buyer
            buyer_id = record.get('buyer', {}).get('id')
            if buyer_id and buyer_id not in self.buyers:
                self.buyers[buyer_id] = {
                    'id': buyer_id,
                    'name': record.get('buyer', {}).get('name', ''),
                    'idx': len(self.buyers)  # Sequential index
                }

            # Extract tender
            tender_id = record.get('tender', {}).get('id')
            if tender_id and tender_id not in self.tenders:
                # Parse dates
                start_date = record.get('tender', {}).get('tenderPeriod', {}).get('startDate')
                end_date = record.get('tender', {}).get('tenderPeriod', {}).get('endDate')

                self.tenders[tender_id] = {
                    'id': tender_id,
                    'title': record.get('tender', {}).get('title', ''),
                    'value': record.get('tender', {}).get('value', {}).get('amount', 0),
                    'status': record.get('tender', {}).get('status', ''),
                    'method': record.get('tender', {}).get('procurementMethod', ''),
                    'start_date': self._parse_date(start_date),
                    'end_date': self._parse_date(end_date),
                    'idx': len(self.tenders)
                }

    def _create_edges(self, records: List[Dict]):
        """Create edges between buyers and tenders."""

        for record in records:
            buyer_id = record.get('buyer', {}).get('id')
            tender_id = record.get('tender', {}).get('id')

            if buyer_id in self.buyers and tender_id in self.tenders:
                buyer_idx = self.buyers[buyer_id]['idx']
                tender_idx = self.tenders[tender_id]['idx']

                # Edge: Buyer issues Tender
                start_date = record.get('tender', {}).get('tenderPeriod', {}).get('startDate')

                self.edges[('Buyer', 'issues', 'Tender')].append({
                    'src': buyer_idx,
                    'dst': tender_idx,
                    'timestamp': self._parse_date(start_date)
                })

    def _to_hetero_data(self) -> HeteroData:
        """Convert to PyTorch Geometric HeteroData format."""

        data = HeteroData()

        # Buyer node features
        buyer_features = []
        for buyer_id in sorted(self.buyers.keys(), key=lambda x: self.buyers[x]['idx']):
            buyer = self.buyers[buyer_id]
            # Simple features: one-hot encoding for now
            # In full implementation: embeddings, historical stats, etc.
            features = [1.0, 0.0]  # Placeholder: [is_buyer, is_tender]
            buyer_features.append(features)

        data['Buyer'].x = torch.FloatTensor(buyer_features)
        data['Buyer'].num_nodes = len(self.buyers)

        # Tender node features
        tender_features = []
        tender_values = []
        tender_timestamps = []

        for tender_id in sorted(self.tenders.keys(), key=lambda x: self.tenders[x]['idx']):
            tender = self.tenders[tender_id]

            # Normalize tender value (log scale to handle large values)
            value = tender['value']
            log_value = np.log10(value + 1)  # +1 to avoid log(0)

            features = [0.0, 1.0, log_value]  # [is_buyer, is_tender, log_value]
            tender_features.append(features)
            tender_values.append(value)

            # Store timestamp as days since epoch
            if tender['start_date']:
                # Make the reference datetime timezone-aware (UTC)
                days_since_epoch = (tender['start_date'] - datetime(2000, 1, 1, tzinfo=timezone.utc)).days
                tender_timestamps.append(days_since_epoch)
            else:
                tender_timestamps.append(0)

        data['Tender'].x = torch.FloatTensor(tender_features)
        data['Tender'].num_nodes = len(self.tenders)
        data['Tender'].value = torch.FloatTensor(tender_values)  # Store for later use
        data['Tender'].timestamp = torch.LongTensor(tender_timestamps)

        # Edges: Buyer issues Tender
        edge_type = ('Buyer', 'issues', 'Tender')
        if edge_type in self.edges:
            edges = self.edges[edge_type]

            src_indices = [e['src'] for e in edges]
            dst_indices = [e['dst'] for e in edges]
            timestamps = []

            for e in edges:
                if e['timestamp']:
                    # Make the reference datetime timezone-aware (UTC)
                    days = (e['timestamp'] - datetime(2000, 1, 1, tzinfo=timezone.utc)).days
                    timestamps.append(days)
                else:
                    timestamps.append(0)

            data[edge_type].edge_index = torch.LongTensor([src_indices, dst_indices])
            data[edge_type].edge_attr = torch.FloatTensor(timestamps).unsqueeze(1)

        return data

    def _parse_date(self, date_str: str) -> datetime:
        """Parse ISO date string."""
        if not date_str:
            return None
        try:
            return datetime.fromisoformat(date_str.replace('Z', '+00:00'))
        except:
            return None


# ============================================================================
# PART 3: SYNTHETIC FRAUD INJECTION
# ============================================================================

class FraudInjector:
    """
    Injects synthetic fraud patterns into procurement graph.

    Fraud patterns:
    1. Burst fraud: Buyer suddenly issues many high-value tenders
    2. Round amounts: Tenders with suspiciously round values
    3. Rapid sequences: Multiple tenders issued in short time
    """
    
    def __init__(self, graph: HeteroData):
        self.graph = graph
        self.fraud_labels = []

    def inject_burst_fraud(self, num_buyers: int = 10, burst_size: int = 5) -> HeteroData:
        """
        Inject burst fraud: Buyer suddenly issues many tenders.

        Pattern:
        - Select random buyer
        - Add multiple high-value tenders in short time window
        - Mark as fraud
        """
        print(f"Injecting burst fraud: {num_buyers} buyers, {burst_size} tenders each")

        num_buyers_total = self.graph['Buyer'].num_nodes
        num_tenders_total = self.graph['Tender'].num_nodes

        for _ in range(num_buyers):
            # Select random buyer
            buyer_idx = np.random.randint(0, num_buyers_total)

            # Create burst of tenders
            for _ in range(burst_size):
                # Create new tender node
                tender_idx = num_tenders_total
                num_tenders_total += 1

                # High value (suspicious)
                fraud_value = np.random.uniform(1e8, 5e8)  # 100M - 500M COP
                log_value = np.log10(fraud_value + 1)

                # Recent timestamp
                recent_days = np.random.randint(8000, 8100)  # ~2022

                # Add tender features
                new_tender_features = torch.FloatTensor([[0.0, 1.0, log_value]])
                self.graph['Tender'].x = torch.cat([
                    self.graph['Tender'].x,
                    new_tender_features
                ], dim=0)

                # Add edge
                edge_type = ('Buyer', 'issues', 'Tender')
                new_edge = torch.LongTensor([[buyer_idx], [tender_idx]])
                new_edge_attr = torch.FloatTensor([[recent_days]])

                if edge_type in self.graph.edge_types:
                    self.graph[edge_type].edge_index = torch.cat([
                        self.graph[edge_type].edge_index,
                        new_edge
                    ], dim=1)
                    self.graph[edge_type].edge_attr = torch.cat([
                        self.graph[edge_type].edge_attr,
                        new_edge_attr
                    ], dim=0)

                # Mark as fraud
                self.fraud_labels.append({
                    'type': 'BURST',
                    'buyer_idx': buyer_idx,
                    'tender_idx': tender_idx,
                    'value': fraud_value
                })

        # Update node count
        self.graph['Tender'].num_nodes = num_tenders_total

        # Create fraud label tensor
        fraud_mask = torch.zeros(num_tenders_total, dtype=torch.bool)
        for label in self.fraud_labels:
            fraud_mask[label['tender_idx']] = True

        self.graph['Tender'].fraud_label = fraud_mask

        print(f"Injected {len(self.fraud_labels)} fraudulent tenders")
        return self.graph

    def inject_round_amounts(self, num_tenders: int = 20) -> HeteroData:
        """
        Inject tenders with suspiciously round amounts.

        Pattern: Values like exactly 100M, 200M, 500M (no decimals)
        """
        num_tenders_total = self.graph['Tender'].num_nodes
        num_buyers = self.graph['Buyer'].num_nodes

        for _ in range(num_tenders):
            # Random buyer
            buyer_idx = np.random.randint(0, num_buyers)

            # Suspiciously round amount
            round_values = [100e6, 200e6, 500e6, 1e9]  # 100M, 200M, 500M, 1B
            fraud_value = np.random.choice(round_values)
            log_value = np.log10(fraud_value + 1)

            # Create tender
            tender_idx = num_tenders_total
            num_tenders_total += 1

            new_tender_features = torch.FloatTensor([[0.0, 1.0, log_value]])
            self.graph['Tender'].x = torch.cat([
                self.graph['Tender'].x,
                new_tender_features
            ], dim=0)

            # Add edge
            edge_type = ('Buyer', 'issues', 'Tender')
            new_edge = torch.LongTensor([[buyer_idx], [tender_idx]])
            recent_days = np.random.randint(8000, 8100)
            new_edge_attr = torch.FloatTensor([[recent_days]])

            self.graph[edge_type].edge_index = torch.cat([
                self.graph[edge_type].edge_index,
                new_edge
            ], dim=1)
            self.graph[edge_type].edge_attr = torch.cat([
                self.graph[edge_type].edge_attr,
                new_edge_attr
            ], dim=0)

            self.fraud_labels.append({
                'type': 'ROUND_AMOUNT',
                'buyer_idx': buyer_idx,
                'tender_idx': tender_idx,
                'value': fraud_value
            })

        self.graph['Tender'].num_nodes = num_tenders_total

        # Update fraud labels
        fraud_mask = torch.zeros(num_tenders_total, dtype=torch.bool)
        for label in self.fraud_labels:
            fraud_mask[label['tender_idx']] = True

        self.graph['Tender'].fraud_label = fraud_mask

        return self.graph


# PART 4: USAGE

def main():
    """
    Complete pipeline: Load OCD data → Build graph → Inject fraud
    """

    # Step 1: Load OCD data
    print("=======")
    print("STEP 1: Loading OCD Data")
    print("=======")

    loader = OCDDataLoader('data/raw/colombia_procurement_sample.jsonl')
    records = loader.load_data(max_records=MAX_RECORDS)  # Driven by MAX_RECORDS from cell 2

    # Convert to DataFrame for inspection
    df = loader.to_dataframe()
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nSample data:")
    print(df[['tender_id', 'buyer_name', 'tender_value', 'start_date']].head())

    # Step 2: Build graph
    print("=======")
    print("STEP 2: Building Heterogeneous Graph")
    print("=======")

    builder = ProcurementGraphBuilder()
    graph = builder.build_graph(records)

    print(f"\nGraph structure:")
    print(f"  Node types: {graph.node_types}")
    print(f"  Edge types: {graph.edge_types}")
    print(f"  Buyer features shape: {graph['Buyer'].x.shape}")
    print(f"  Tender features shape: {graph['Tender'].x.shape}")

    # Step 3: Inject fraud
    print("=======")
    print("STEP 3: Injecting Synthetic Fraud")
    print("=======")

    injector = FraudInjector(graph)
    # Hold fraud rate at ~6.54% to ensure it's detectable but not overwhelming
    num_burst_buyers = int(0.01 * MAX_RECORDS)  
    num_round        = int(0.02 * MAX_RECORDS) 
    graph = injector.inject_burst_fraud(num_buyers=num_burst_buyers, burst_size=5)
    graph = injector.inject_round_amounts(num_tenders=num_round)

    print(f"\nFraud statistics:")
    print(f"  Total tenders: {graph['Tender'].num_nodes}")
    print(f"  Fraudulent tenders: {graph['Tender'].fraud_label.sum().item()}")
    print(f"  Fraud rate: {graph['Tender'].fraud_label.float().mean().item():.2%}")

    # Step 4: Save graph
    print("=======")
    print("STEP 4: Saving Graph")
    print("=======")

    # Create the directories if they don't exist
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    output_path = 'data/processed/procurement_graph.pt'
    torch.save(graph, output_path)
    print(f"Graph saved to: {output_path}")

    # Save fraud labels separately for evaluation
    fraud_info = {
        'labels': injector.fraud_labels,
        'fraud_mask': graph['Tender'].fraud_label,
        'num_fraud': graph['Tender'].fraud_label.sum().item(),
        'num_total': graph['Tender'].num_nodes
    }
    torch.save(fraud_info, 'data/processed/fraud_labels.pt')
    print(f"Fraud labels saved to: data/processed/fraud_labels.pt")

    print("=======")
    print("PIPELINE COMPLETE!")
    print("=======")
    print("\nNext steps:")
    print("1. Train TA-HGMAE model on this graph")
    print("2. Evaluate anomaly detection performance")
    print("3. Compare with baselines (Isolation Forest, GraphMAE)")

    return graph, injector.fraud_labels


if __name__ == "__main__":
    graph, fraud_labels = main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
import numpy as np
from typing import Dict, List, Tuple
from datetime import datetime, timedelta, timezone

# PART 1: TEMPORAL FEATURE EXTRACTION 

class TemporalFeatureExtractor:
    """
    Extracts 19-dim temporal features per edge:
      - sinusoidal encoding of Δt at 8 frequencies  -> 16 dims
      - exponential recency decay                   ->  1 dim
      - velocity  (recent / historical edge count)  ->  1 dim
      - burst     (std of inter-event intervals)    ->  1 dim
    """

    def __init__(self, window_days: int = 180, num_frequencies: int = 8):
        self.window_days = window_days
        self.num_frequencies = num_frequencies

    def sinusoidal_encoding(self, delta_t_days: np.ndarray) -> np.ndarray:
        freqs = np.logspace(
            np.log10(1.0), np.log10(365.0), num=self.num_frequencies
        )
        encodings = []
        for freq in freqs:
            angle = 2 * np.pi * delta_t_days / freq
            encodings.append(np.sin(angle))
            encodings.append(np.cos(angle))
        return np.column_stack(encodings)

    def compute_recency(self, edge_times: np.ndarray, current_time: datetime) -> np.ndarray:
        delta_t = np.array([(current_time - t).days for t in edge_times])
        sin_encoding = self.sinusoidal_encoding(delta_t)
        decay = np.exp(-0.01 * delta_t).reshape(-1, 1)
        return np.concatenate([sin_encoding, decay], axis=1)

    def compute_velocity(self, u_nodes, v_nodes, edge_index_time, current_time):
        velocities = []
        for u, v in zip(u_nodes, v_nodes):
            edge_key = (u, v)
            if edge_key not in edge_index_time:
                velocities.append(0.0)
                continue
            times = edge_index_time[edge_key]
            recent_cutoff = current_time - timedelta(days=30)
            recent_count = sum(1 for t in times if t >= recent_cutoff)
            hist_cutoff = current_time - timedelta(days=180)
            hist_count = sum(1 for t in times if t >= hist_cutoff)
            velocities.append(recent_count / (hist_count + 1.0))
        return np.array(velocities).reshape(-1, 1)

    def compute_burst(self, u_nodes, v_nodes, edge_index_time):
        bursts = []
        for u, v in zip(u_nodes, v_nodes):
            edge_key = (u, v)
            if edge_key not in edge_index_time or len(edge_index_time[edge_key]) < 2:
                bursts.append(0.0)
                continue
            times = sorted(edge_index_time[edge_key])
            intervals = [(times[i + 1] - times[i]).days for i in range(len(times) - 1)]
            burst_score = np.std(intervals) if len(intervals) > 1 else 0.0
            bursts.append(burst_score)
        return np.array(bursts).reshape(-1, 1)

    def extract_all_features(self, edge_data: Dict, current_time: datetime) -> torch.Tensor:
        base_datetime = datetime(2000, 1, 1, tzinfo=timezone.utc)
        edge_datetimes = [
            base_datetime + timedelta(days=int(d.item()))
            for d in edge_data['times']
        ]
        recency = self.compute_recency(edge_datetimes, current_time)
        velocity = self.compute_velocity(
            edge_data['u_nodes'], edge_data['v_nodes'],
            edge_data['edge_index_time'], current_time
        )
        burst = self.compute_burst(
            edge_data['u_nodes'], edge_data['v_nodes'], edge_data['edge_index_time']
        )
        temporal_features = np.concatenate([recency, velocity, burst], axis=1)
        return torch.FloatTensor(temporal_features)

# PART 2: HGT ENCODER  

class HGTEncoder(nn.Module):
    """Heterogeneous Graph Transformer encoder (H-dimension of HTS)."""

    def __init__(self, node_types, edge_types, hidden_dim=128, num_layers=2, num_heads=4):
        super().__init__()
        self.node_types = node_types
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.node_embeddings = nn.ModuleDict({
            ntype: Linear(-1, hidden_dim) for ntype in node_types
        })

        self.convs = nn.ModuleList([
            HGTConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                metadata=(node_types, edge_types),
                heads=num_heads,
            )
            for _ in range(num_layers)
        ])

        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])

    def forward(self, x_dict, edge_index_dict):
        h_dict = {
            ntype: self.node_embeddings[ntype](x_dict[ntype])
            for ntype in self.node_types
        }
        for conv, norm in zip(self.convs, self.norms):
            h_dict_new = conv(h_dict, edge_index_dict)
            next_h_dict = {}
            for ntype in self.node_types:
                if ntype in h_dict_new:
                    next_h_dict[ntype] = norm(h_dict[ntype] + h_dict_new[ntype])
                else:
                    next_h_dict[ntype] = norm(h_dict[ntype])
            h_dict = next_h_dict
        return h_dict


# PART 3: TA-HGMAE (OVERALL MODEL ARCHITECTURE) — with lazy input projections to handle enriched features without dim issues.

class TA_HGMAE(nn.Module):
    """
    Temporal-Aware Heterogeneous Graph Masked Autoencoder.
    """

    def __init__(self, node_types, edge_types, node_feat_dims,
             hidden_dim=128, num_layers=2, mask_rate=0.15, window_days=180,
             use_remask=False):
        super().__init__()

        self.node_types = node_types
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.mask_rate = mask_rate

        # Store ORIGINAL feature dims — the decoder reconstructs back to these.
        # (Temporal features are inputs, never targets.)
        self.node_feat_dims = dict(node_feat_dims)

        # Temporal feature extractor (produces 19 dims per edge)
        self.temporal_extractor = TemporalFeatureExtractor(window_days=window_days)

        # Cached temporal feature dim (num_frequencies * 2 + recency + velocity + burst)
        self.temporal_dim = self.temporal_extractor.num_frequencies * 2 + 3

        # HGT encoder (lazy input dim)
        self.encoder = HGTEncoder(
            node_types=node_types,
            edge_types=edge_types,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
        )

        # Type-specific decoders — output dim = ORIGINAL feature dim only.
        self.decoders = nn.ModuleDict({
            ntype: nn.Sequential(
                Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                Linear(hidden_dim, node_feat_dims[ntype]),  
            )
            for ntype in node_types
        })
        self.use_remask = use_remask
        self.decoder_mask_tokens = nn.ParameterDict({
            ntype: nn.Parameter(torch.randn(1, hidden_dim) * 0.02)
            for ntype in node_types
        })

        # (Unused in current loss but kept for schema completeness)
        self.edge_decoder = nn.Sequential(
            Linear(hidden_dim * 2, hidden_dim), nn.ReLU(),
            Linear(hidden_dim, 1), nn.Sigmoid(),
        )
        self.temporal_decoder = nn.Sequential(
            Linear(hidden_dim * 2, hidden_dim), nn.ReLU(),
            Linear(hidden_dim, 1),
        )
    # helpers

    def _extract_edge_temporal_features(self, data, current_time):
        """Per-edge 19-dim temporal features for every edge type."""
        edge_temporal = {}
        for edge_type in self.edge_types:
            if (edge_type not in data.edge_index_dict
                    or data[edge_type].edge_index.numel() == 0):
                edge_temporal[edge_type] = torch.empty(0, self.temporal_dim)
                continue

            edge_index = data[edge_type].edge_index
            if (hasattr(data[edge_type], 'edge_attr')
                    and data[edge_type].edge_attr is not None
                    and data[edge_type].edge_attr.numel() > 0):
                edge_times = data[edge_type].edge_attr.squeeze()
            else:
                # Fallback: treat all edges as happening at current_time
                days_since_base = (
                    current_time - datetime(2000, 1, 1, tzinfo=timezone.utc)
                ).days
                edge_times = torch.full(
                    (edge_index.shape[1],), days_since_base, dtype=torch.float32
                )

            edge_data = {
                'times': edge_times,
                'u_nodes': edge_index[0].cpu().numpy(),
                'v_nodes': edge_index[1].cpu().numpy(),
                'edge_index_time': {},
            }

            edge_temporal[edge_type] = self.temporal_extractor.extract_all_features(
                edge_data, current_time
            )
        return edge_temporal

    def _aggregate_temporal_to_nodes(self, edge_temporal_features, data):
        """
        Aggregate per-edge temporal features to per-node features via mean
        pooling over every edge the node participates in (as src or dst).
          - each Buyer gets the mean of its OUTGOING edges' temporal vectors
          - each Tender gets the mean of its INCOMING edges' temporal vectors
        """
        node_temp = {}
        node_counts = {}
        for ntype in self.node_types:
            n_nodes = data[ntype].num_nodes
            node_temp[ntype] = torch.zeros(n_nodes, self.temporal_dim)
            node_counts[ntype] = torch.zeros(n_nodes, 1)

        for edge_type, temp_feat in edge_temporal_features.items():
            if temp_feat.numel() == 0:
                continue

            src_type, _, dst_type = edge_type
            edge_index = data[edge_type].edge_index  # [2, E]
            num_edges = edge_index.shape[1]
            ones = torch.ones(num_edges, 1)

            # Scatter-add to source nodes (outgoing-edge aggregation)
            node_temp[src_type].index_add_(0, edge_index[0], temp_feat)
            node_counts[src_type].index_add_(0, edge_index[0], ones)

            # Scatter-add to target nodes (incoming-edge aggregation)
            node_temp[dst_type].index_add_(0, edge_index[1], temp_feat)
            node_counts[dst_type].index_add_(0, edge_index[1], ones)

        # Mean = sum / count, with a small epsilon to guard orphan nodes
        for ntype in self.node_types:
            node_temp[ntype] = node_temp[ntype] / (node_counts[ntype] + 1e-8)

        return node_temp

    def mask_features(self, enriched_x_dict):
        """
        Randomly mask whole-node ORIGINAL features (GraphMAE-style).
        Temporal context stays visible so the encoder can still condition on it.

        enriched_x_dict: {ntype: [N, orig_dim + temporal_dim]}
        Returns masked enriched tensors and the per-node boolean masks.
        """
        masked_x_dict = {}
        masks = {}
        for ntype, x in enriched_x_dict.items():
            if x.numel() == 0:
                masks[ntype] = torch.empty(0, dtype=torch.bool)
                masked_x_dict[ntype] = x.clone()
                continue

            mask = torch.rand(x.shape[0]) < self.mask_rate
            masked_x = x.clone()
            orig_dim = self.node_feat_dims[ntype]
            if mask.any():
                masked_x[mask, :orig_dim] = 0.0

            masked_x_dict[ntype] = masked_x
            masks[ntype] = mask
        return masked_x_dict, masks

    # Forward / loss / scoring

    def forward(self, data: HeteroData, current_time: datetime) -> Dict:
        # 1. Per-edge temporal features (19-dim)
        edge_temporal = self._extract_edge_temporal_features(data, current_time)

        # 2. Aggregate to per-node temporal context via mean pooling
        node_temporal = self._aggregate_temporal_to_nodes(edge_temporal, data)

        # 3. Concatenate [original_features ‖ temporal_context] BEFORE masking
        enriched_x_dict = {}
        for ntype in self.node_types:
            enriched_x_dict[ntype] = torch.cat(
                [data.x_dict[ntype], node_temporal[ntype]], dim=-1
            )

        # 4. Mask only the ORIGINAL feature block; temporal stays visible
        masked_x_dict, feature_masks = self.mask_features(enriched_x_dict)

        # 5. Encode with HGT (lazy projections adapt to the wider input)
        embeddings = self.encoder(masked_x_dict, data.edge_index_dict)

        # 5b. (Optional) GraphMAE-style re-mask decoding.
        # Substitute encoder outputs at masked positions with the learnable [MASK]
        # token so the decoder cannot just read off the encoder's self-encoding of
        # the masked node -- it has to predict from neighbour context.
        if self.use_remask:
            embeddings_for_decoder = {}
            for ntype in self.node_types:
                h = embeddings[ntype]
                if ntype in feature_masks and feature_masks[ntype].any():
                    h = h.clone()
                    h[feature_masks[ntype]] = self.decoder_mask_tokens[ntype]
                embeddings_for_decoder[ntype] = h
        else:
            embeddings_for_decoder = embeddings

        # 6. Decode -- reconstructs ORIGINAL features only (temporal not a target)
        reconstructed_features = {
            ntype: self.decoders[ntype](embeddings_for_decoder[ntype])
            for ntype in self.node_types
        }

        return {
            'embeddings': embeddings,
            'reconstructed_features': reconstructed_features,
            'feature_masks': feature_masks,
            'temporal_features': edge_temporal,           # per-edge
            'node_temporal_features': node_temporal,      # per-node (for inspection/debug)
        }

    def compute_loss(self, output, data):
        losses = {}

        # Loss 1: Feature reconstruction on masked nodes (scaled cosine error)
        feat_loss = 0.0
        for ntype in self.node_types:
            if (ntype not in output['feature_masks']
                    or not output['feature_masks'][ntype].any()):
                continue
            mask = output['feature_masks'][ntype]
            if mask.sum() == 0:
                continue

            # Targets are ORIGINAL features (data.x_dict), NOT the enriched version
            original = data.x_dict[ntype][mask]
            reconstructed = output['reconstructed_features'][ntype][mask]

            if original.numel() > 0 and reconstructed.numel() > 0:
                cos_sim = F.cosine_similarity(original, reconstructed, dim=1)
                feat_loss = feat_loss + (1 - cos_sim).mean()

        losses['feature_reconstruction'] = feat_loss
        losses['structural'] = torch.tensor(0.0)
        losses['temporal'] = torch.tensor(0.0)

        losses['total'] = (
            0.4 * losses['feature_reconstruction']
            + 0.4 * losses['structural']
            + 0.2 * losses['temporal']
        )
        return losses

    def anomaly_score(self, data, current_time):
        """
        Reconstruction-error anomaly score per node.

        Fix 1: the internal forward() call now runs through the temporal-enrichment
        path, so reconstruction error reflects a model that actually saw temporal
        context. Error is measured on the ORIGINAL feature block only.
        """
        self.eval()
        with torch.no_grad():
            output = self.forward(data, current_time)
            scores = {}
            for ntype in self.node_types:
                original = data.x_dict[ntype]                       # [N, orig_dim]
                reconstructed = output['reconstructed_features'][ntype]  # [N, orig_dim]
                # Per-node MSE over original features
                error = ((original - reconstructed) ** 2).mean(dim=1)
                scores[ntype] = error
        return scores


# PART 4: TRAINING UTILITIES 

def train_ta_hgmae(model, data, num_epochs=100, lr=0.001, current_time=None):
    if current_time is None:
        current_time = datetime.now(timezone.utc)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = model(data, current_time)
        losses = model.compute_loss(output, data)
        losses['total'].backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {losses['total'].item():.4f}")

    return model

# PART 5:  USAGE

if __name__ == "__main__":
    node_types = ['Buyer', 'Tender']
    edge_types = [('Buyer', 'issues', 'Tender')]

    try:
        graph = torch.load('data/processed/procurement_graph.pt', weights_only=False)
        fraud_labels_info = torch.load('data/processed/fraud_labels.pt', weights_only=False)
        fraud_labels = fraud_labels_info['labels']
    except FileNotFoundError:
        print("Error: procurement_graph.pt or fraud_labels.pt not found. Run the data cell first.")
        raise

    node_feat_dims = {
        'Buyer': graph['Buyer'].x.shape[1],
        'Tender': graph['Tender'].x.shape[1],
    }
    print(f"Inferred node feature dimensions: {node_feat_dims}")

    model = TA_HGMAE(
        node_types=node_types,
        edge_types=edge_types,
        node_feat_dims=node_feat_dims,
        hidden_dim=128,
        num_layers=2,
        mask_rate=0.15,
        window_days=180,
    )

    # Dummy forward pass to trigger lazy initialisation of the wider input projections
    dummy_current_time = datetime.now(timezone.utc)
    _ = model(graph, dummy_current_time)
    print("Lazy layers initialised.")

    print("TA-HGMAE Model initialised.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

    print("\n" + "==========")
    print("STEP 4: Training TA-HGMAE Model")
    print("=================")
    trained_model = train_ta_hgmae(model, graph, num_epochs=100)

    print("\n" + "==========")
    print("STEP 5: Detecting Anomalies")
    print("==========")
    scores = trained_model.anomaly_score(graph, current_time=datetime.now(timezone.utc))
    print("Sample Tender anomaly scores:", scores['Tender'][:5])
    top = torch.topk(scores['Tender'], k=10)
    print(f"Top 10 anomalous Tender indices: {top.indices}")
    print(f"Top 10 anomalous Tender scores:  {top.values}")


In [ ]:
import torch
import torch.nn.functional as F
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, roc_auc_score
import numpy as np
from datetime import datetime, timezone
from torch_geometric.nn import GCNConv

# Baseline 1: Isolation Forest (tabular, unchanged)

class IsolationForestBaseline:
    def __init__(self, contamination=0.1):
        self.model = IsolationForest(
            contamination=contamination, random_state=42, n_estimators=100
        )
        self.fitted = False

    def fit(self, data):
        X = data['Tender'].x.numpy()
        self.model.fit(X)
        self.fitted = True
        return self

    def predict_scores(self, data):
        if not self.fitted:
            raise ValueError("Model not fitted.")
        X = data['Tender'].x.numpy()
        scores = -self.model.decision_function(X)
        return torch.FloatTensor(scores)


# Baseline 2: Tabular Autoencoder (unchanged)

class TabularAutoencoder(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=32):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim), torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim // 2), torch.nn.ReLU(),
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim // 2, hidden_dim), torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, in_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def fit(self, data, num_epochs=100, lr=0.001):
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        X = data['Tender'].x
        print("Training Tabular Autoencoder")
        for epoch in range(num_epochs):
            optimizer.zero_grad()
            loss = F.mse_loss(self.forward(X), X)
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")
        print(" Training complete")
        return self

    def predict_scores(self, data):
        self.eval()
        with torch.no_grad():
            X = data['Tender'].x
            x_recon = self.forward(X)
            return ((X - x_recon) ** 2).mean(dim=1)

# Baseline 3: Homogeneous GraphMAE 

class HomogeneousGraphMAE(torch.nn.Module):
    """
    GraphMAE on a homogeneous graph with REAL message passing.

    Previous version was an MLP that never touched edge_index, so its
    "graph baseline" label was misleading. This version:
      - pads per-type node features to a uniform dim (required by
        data.to_homogeneous(), because Buyer=2 dims, Tender=3 dims)
      - converts the hetero graph via data.to_homogeneous()
      - adds reverse edges so GCN propagates bidirectionally on the
        bipartite Buyer↔Tender graph
      - applies 2-layer GCNConv encoder + MLP decoder
      - returns per-Tender reconstruction error for fraud evaluation
    """

    def __init__(self, in_dim, hidden_dim=64, mask_rate=0.15):
        super().__init__()
        self.mask_rate = mask_rate
        self.in_dim = in_dim  # padded uniform feature dim across node types

        # Real message-passing encoder 
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # Decoder: reconstruct padded input features
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim), torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, in_dim),
        )


    def _to_homogeneous(self, hetero_data):
        """
        Pad each node type's features to a uniform dim, then convert the
        heterogeneous graph to a homogeneous one. Returns a Data object
        with .x, .edge_index, and .node_type (used later to slice out
        Tender nodes for scoring).
        """
        max_dim = max(hetero_data[nt].x.shape[1] for nt in hetero_data.node_types)

        hetero_padded = hetero_data.clone()
        for nt in hetero_padded.node_types:
            x = hetero_padded[nt].x
            pad = max_dim - x.shape[1]
            if pad > 0:
                hetero_padded[nt].x = torch.cat(
                    [x, torch.zeros(x.shape[0], pad)], dim=1
                )

        homo = hetero_padded.to_homogeneous()

        # Make undirected: GCN on the bipartite graph would otherwise only
        # push messages Buyer→Tender. Adding flipped edges lets information
        # flow both ways, which is the norm for GraphMAE.
        ei = homo.edge_index
        homo.edge_index = torch.cat([ei, ei.flip(0)], dim=1)

        return homo


    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = self.conv2(h, edge_index)
        return h

    def mask_features(self, x):
        mask = torch.rand(x.shape[0]) < self.mask_rate
        x_masked = x.clone()
        x_masked[mask] = 0
        return x_masked, mask

    def forward(self, homo):
        x_masked, mask = self.mask_features(homo.x)
        z = self.encode(x_masked, homo.edge_index)
        x_recon = self.decoder(z)
        return x_recon, mask

    def fit(self, data, num_epochs=100, lr=0.001):
        homo = self._to_homogeneous(data)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        print("Training Homogeneous GraphMAE (GCNConv)...")
        for epoch in range(num_epochs):
            self.train()
            optimizer.zero_grad()
            x_recon, mask = self.forward(homo)
            if mask.sum() == 0:
                continue
            loss = F.mse_loss(x_recon[mask], homo.x[mask])
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")
        print("  ✓ Training complete")
        return self

    def predict_scores(self, data):
        """Returns per-Tender reconstruction error (matches evaluator convention)."""
        self.eval()
        with torch.no_grad():
            homo = self._to_homogeneous(data)
            z = self.encode(homo.x, homo.edge_index)
            x_recon = self.decoder(z)
            err_all = ((homo.x - x_recon) ** 2).mean(dim=1)

            # Slice out Tender nodes. to_homogeneous() orders nodes in the
            # same order as data.node_types, so node_type ids are consistent.
            tender_type_idx = list(data.node_types).index('Tender')
            tender_mask = (homo.node_type == tender_type_idx)
            tender_scores = err_all[tender_mask]
        return tender_scores


# Evaluation Functions  (unchanged)

def precision_at_k(scores, labels, k=100):
    if k > len(scores):
        k = len(scores)
    top_k_idx = torch.topk(scores, k=k).indices
    return labels[top_k_idx].sum().item() / k


def evaluate_all_models(models_dict, data, fraud_labels):
    results = []
    print("=======")
    print("EVALUATING ALL MODELS")
    print("========")

    for model_name, model in models_dict.items():
        print(f"Evaluating: {model_name}")
        print("--------")

        scores = model.predict_scores(data)
        scores_np = scores.numpy()
        labels_np = fraud_labels.numpy().astype(int)

        precision_at_50 = precision_at_k(scores, fraud_labels, k=50)
        precision_at_100 = precision_at_k(scores, fraud_labels, k=100)
        precision_at_200 = precision_at_k(scores, fraud_labels, k=200)

        try:
            auc = roc_auc_score(labels_np, scores_np)
        except Exception:
            auc = 0.0

        top_100_idx = torch.topk(scores, k=min(100, len(scores))).indices
        detection_rate = fraud_labels[top_100_idx].float().mean().item()

        results.append({
            'Model': model_name,
            'P@50': precision_at_50,
            'P@100': precision_at_100,
            'P@200': precision_at_200,
            'AUC-ROC': auc,
            'Detection Rate': detection_rate,
        })

        print(f"  Precision@50:  {precision_at_50:.3f}")
        print(f"  Precision@100: {precision_at_100:.3f}")
        print(f"  Precision@200: {precision_at_200:.3f}")
        print(f"  AUC-ROC:       {auc:.3f}")
        print(f"  Detection:     {detection_rate:.1%}\n")

    import pandas as pd
    results_df = pd.DataFrame(results)

    print("=======")
    print("SUMMARY TABLE")
    print(results_df.to_string(index=False))
    print()
    return results_df


# Wrapper + comparison entry point


class ModelWrapper:
    def __init__(self, model, current_time):
        self.model = model
        self.current_time = current_time

    def predict_scores(self, data):
        return self.model.anomaly_score(data, self.current_time)['Tender']


def run_full_comparison(data, fraud_labels):
    node_types = list(data.node_types)
    edge_types = list(data.edge_types)
    node_feat_dims = {nt: data[nt].x.shape[1] for nt in node_types}

    in_dim_tender = data['Tender'].x.shape[1]
    # The homogeneous baseline operates on padded features, so it needs
    # the max feature dim across node types, not just Tender's.
    max_feat_dim = max(data[nt].x.shape[1] for nt in node_types)

    print("=======")
    print("TRAINING ALL MODELS")
    print("=" * 70 + "\n")

    models = {}

    # 1. Isolation Forest
    print("1. Isolation Forest (Tabular)")
    print("-" * 70)
    models['Isolation Forest'] = IsolationForestBaseline(
        contamination=fraud_labels.float().mean().item()
    )
    models['Isolation Forest'].fit(data)
    print("  ✓ Fitted\n")

    # 2. Tabular Autoencoder
    print("2. Tabular Autoencoder")
    print("-" * 70)
    models['Tabular AE'] = TabularAutoencoder(in_dim=in_dim_tender, hidden_dim=32)
    models['Tabular AE'].fit(data, num_epochs=100, lr=0.001)
    print()

    # 3. Homogeneous GraphMAE (now real GCN — Fix 2)
    print("3. Homogeneous GraphMAE (GCNConv)")
    print("-" * 70)
    models['GraphMAE (Homo)'] = HomogeneousGraphMAE(
        in_dim=max_feat_dim,   # padded uniform dim, not in_dim_tender
        hidden_dim=64,
        mask_rate=0.15,
    )
    models['GraphMAE (Homo)'].fit(data, num_epochs=100, lr=0.001)
    print()

    # 4. TA-HGMAE (now with temporal wiring — Fix 1)
    print("4. TA-HGMAE ")
    model = TA_HGMAE(
        node_types=node_types,
        edge_types=edge_types,
        node_feat_dims=node_feat_dims,
        hidden_dim=32,
        num_layers=2,
        mask_rate=0.15,
        window_days=180,
    )
    print("  Training TA-HGMAE...")
    trained_model_tahgmae = train_ta_hgmae(model, data, num_epochs=100)
    print("  Training complete")
    models['TA-HGMAE'] = ModelWrapper(
        trained_model_tahgmae, datetime.now(timezone.utc)
    )
    print()

    results_df = evaluate_all_models(models, data, fraud_labels)
    return models, results_df


# ----------------------------------------------------------------------------
# Entry point
# ----------------------------------------------------------------------------

if __name__ == "__main__":
    data = torch.load('data/processed/procurement_graph.pt', weights_only=False)
    fraud_labels = data['Tender'].fraud_label

    print(f"Dataset loaded:")
    print(f"  Total tenders: {data['Tender'].num_nodes}")
    print(f"  Fraud tenders: {fraud_labels.sum().item()}")
    print(f"  Fraud rate:    {fraud_labels.float().mean().item():.2%}\n")

    models, results = run_full_comparison(data, fraud_labels)
    results.to_csv('results/baseline_comparison.csv', index=False)
    print("Results saved to: results/baseline_comparison.csv")


In [ ]:
# EXPLAINABILITY MODULE (Objective 3)
# Implements:
#   1. Attention-based edge explanation (Eq. 13)
#   2. Residual-based feature explanation (Eq. 14)
#   3. Combined explanation generator
#   4. Explanation Precision@K evaluation

import torch
import torch.nn.functional as F
import numpy as np
from datetime import datetime, timezone


class AttentionCapture:
    """Computes attention proxy scores from trained embeddings."""

    def __init__(self):
        self.attention_weights = {}

    def compute_attention(self, model, data, current_time):
        """
        Attention proxy via cosine similarity between source and destination
        embeddings. After training, nodes that strongly influenced each
        other's reconstruction have similar embeddings in HGT's space.
        """
        model.eval()
        with torch.no_grad():
            output = model.forward(data, current_time)
            embeddings = output['embeddings']

        attention_scores = {}
        for edge_type in model.edge_types:
            if edge_type not in data.edge_index_dict:
                continue
            edge_index = data[edge_type].edge_index
            src_type, _, dst_type = edge_type

            h_src = embeddings[src_type][edge_index[0]]
            h_dst = embeddings[dst_type][edge_index[1]]

            cos_sim = F.cosine_similarity(h_src, h_dst, dim=1)
            cos_sim = (cos_sim - cos_sim.min()) / (cos_sim.max() - cos_sim.min() + 1e-8)
            attention_scores[edge_type] = cos_sim

        self.attention_weights = attention_scores
        return attention_scores


class ResidualExplainer:
    """
    Per-feature z-score explanation.
    z_ij = (r_ij - mu_j) / sigma_j where r_ij = (x_ij - x_hat_ij)^2
    """

    def __init__(self, feature_names=None):
        self.training_mean = None
        self.training_std = None
        self.feature_names = feature_names or {}

    def fit(self, model, data, current_time):
        """Compute per-feature error statistics on training data."""
        model.eval()
        with torch.no_grad():
            output = model.forward(data, current_time)

        self.training_mean = {}
        self.training_std = {}

        for ntype in model.node_types:
            original = data.x_dict[ntype]
            reconstructed = output['reconstructed_features'][ntype]
            residuals = (original - reconstructed) ** 2
            self.training_mean[ntype] = residuals.mean(dim=0)
            self.training_std[ntype] = residuals.std(dim=0) + 1e-8
        return self

    def compute_zscores(self, model, data, current_time, node_type='Tender'):
        """Per-feature z-scores for all nodes."""
        model.eval()
        with torch.no_grad():
            output = model.forward(data, current_time)
        original = data.x_dict[node_type]
        reconstructed = output['reconstructed_features'][node_type]
        residuals = (original - reconstructed) ** 2
        z = (residuals - self.training_mean[node_type]) / self.training_std[node_type]
        return z

    def explain_node(self, z_scores, node_idx, top_k=3, node_type='Tender'):
        """Top-K features by z-score for a single node."""
        z = z_scores[node_idx]
        k = min(top_k, len(z))
        values, indices = torch.abs(z).topk(k)
        names = self.feature_names.get(node_type, {})
        results = []
        for val, idx in zip(values.tolist(), indices.tolist()):
            feat_name = names.get(idx, f"feature_{idx}")
            sign = "+" if z[idx].item() > 0 else "-"
            results.append({
                'feature_index': idx,
                'feature_name': feat_name,
                'z_score': z[idx].item(),
                'abs_z': val,
                'readable': f"{feat_name} ({sign}{val:.1f}\u03C3)"
            })
        return results


class ExplainabilityModule:
    """
    Combined explainability: attention edges + residual features.
    Produces structured auditor-facing output per flagged entity.
    """

    def __init__(self, model, data, current_time, top_k_edges=3, top_k_features=3):
        self.model = model
        self.data = data
        self.current_time = current_time
        self.top_k_edges = top_k_edges
        self.top_k_features = top_k_features

        self.feature_names = {
            'Tender': {0: 'procurement_method', 1: 'tender_status', 2: 'log_tender_value'},
            'Buyer': {0: 'buyer_frequency', 1: 'buyer_avg_value'}
        }

        # Build sub-modules
        print("  Fitting training residual distribution...")
        self.residual_explainer = ResidualExplainer(feature_names=self.feature_names)
        self.residual_explainer.fit(model, data, current_time)

        print("  Computing attention proxy scores...")
        self.attention_capture = AttentionCapture()
        self.attention_scores = self.attention_capture.compute_attention(model, data, current_time)

        print("  Computing per-feature z-scores...")
        self.tender_zscores = self.residual_explainer.compute_zscores(
            model, data, current_time, node_type='Tender'
        )

        scores = model.anomaly_score(data, current_time)
        self.tender_scores = scores['Tender']
        print("  \u2713 Explainability module ready")

    def get_percentile(self, score):
        return (self.tender_scores < score).float().mean().item() * 100

    def explain_node(self, tender_idx):
        result = {
            'node_index': tender_idx,
            'anomaly_score': self.tender_scores[tender_idx].item(),
            'percentile': self.get_percentile(self.tender_scores[tender_idx]),
            'top_edges': [],
            'top_features': [],
        }

        # Edge explanation
        for edge_type in self.model.edge_types:
            if edge_type not in self.data.edge_index_dict:
                continue
            edge_index = self.data[edge_type].edge_index
            src_type, rel, dst_type = edge_type

            if dst_type == 'Tender':
                edge_mask = edge_index[1] == tender_idx
                partner_indices = edge_index[0][edge_mask]
                partner_type = src_type
            elif src_type == 'Tender':
                edge_mask = edge_index[0] == tender_idx
                partner_indices = edge_index[1][edge_mask]
                partner_type = dst_type
            else:
                continue

            if not edge_mask.any():
                continue

            attn = self.attention_scores[edge_type][edge_mask]
            k = min(self.top_k_edges, len(attn))
            top_vals, top_idx = attn.topk(k)

            for val, idx in zip(top_vals.tolist(), top_idx.tolist()):
                result['top_edges'].append({
                    'partner_type': partner_type,
                    'partner_index': partner_indices[idx].item(),
                    'relation': rel,
                    'attention': val,
                    'readable': f"{partner_type} {partner_indices[idx].item()} \u2192 {rel} (attn: {val:.3f})"
                })

        # Feature explanation
        result['top_features'] = self.residual_explainer.explain_node(
            self.tender_zscores, tender_idx,
            top_k=self.top_k_features, node_type='Tender'
        )
        return result

    def print_explanation(self, explanation, is_fraud=False):
        e = explanation
        tag = "\u2605 FRAUD" if is_fraud else "  normal"
        print(f"  [{tag}] Tender {e['node_index']}  "
              f"(score: {e['anomaly_score']:.4f}, percentile: {e['percentile']:.1f})")
        if e['top_edges']:
            print(f"    Relationships:")
            for edge in e['top_edges']:
                print(f"      {edge['readable']}")
        if e['top_features']:
            print(f"    Deviations:")
            for feat in e['top_features']:
                print(f"      {feat['readable']}")
        print()


def evaluate_explanation_precision(explainer, fraud_labels, top_k=3):
    """
    Explanation Precision@K: for each known fraud tender, what fraction
    of the top-K explained features are the actually manipulated ones?
    FraudInjector modifies tender value, so index 2 (log_tender_value) is ground truth.
    """
    injected_features = {2}  # log_tender_value
    fraud_indices = torch.where(fraud_labels == 1)[0].tolist()

    if not fraud_indices:
        print("  No fraud labels found.")
        return {}

    precisions = []
    hits = []

    for idx in fraud_indices:
        exp = explainer.explain_node(idx)
        top_feat_idx = {f['feature_index'] for f in exp['top_features'][:top_k]}
        h = len(top_feat_idx & injected_features)
        denom = min(top_k, len(exp['top_features'])) if exp['top_features'] else 1
        precisions.append(h / denom)
        hits.append(1.0 if h > 0 else 0.0)

    return {
        'num_fraud': len(fraud_indices),
        'avg_precision_at_k': np.mean(precisions),
        'hit_rate': np.mean(hits),
        'top_k': top_k,
    }

# RUN EXPLAINABILITY

# Use the trained model from the previous cell
data = torch.load('data/processed/procurement_graph.pt', weights_only=False)
fraud_labels = data['Tender'].fraud_label
current_time = datetime.now(timezone.utc)

try:
    model = trained_model_tahgmae
    print("Using trained TA-HGMAE from previous cell.\n")
except NameError:
    try:
        model = list(models.values())[-1].model  
        print("Using model from comparison dict.\n")
    except:
        print("ERROR: No trained model found. Run cell 5 first.")
        raise

print("=======")
print("BUILDING EXPLAINABILITY MODULE (Objective 3)")
print("=======")

explainer = ExplainabilityModule(
    model=model, data=data, current_time=current_time,
    top_k_edges=3, top_k_features=3,
)

# Flag top 5%
scores = model.anomaly_score(data, current_time)
tender_scores = scores['Tender']
threshold = torch.quantile(tender_scores, 0.95)
flagged = torch.where(tender_scores > threshold)[0].tolist()
flagged_sorted = sorted(flagged, key=lambda i: tender_scores[i].item(), reverse=True)

num_fraud_flagged = sum(1 for i in flagged_sorted if fraud_labels[i] == 1)
print(f"\nFlagged {len(flagged_sorted)} tenders above 95th percentile")
print(f"Of which {num_fraud_flagged} are known fraud ({num_fraud_flagged}/{len(flagged_sorted)} = {num_fraud_flagged/max(len(flagged_sorted),1):.1%})\n")

# Print top 10 explanations
print("=======")
print("TOP 10 FLAGGED ENTITIES WITH EXPLANATIONS")
print()

for idx in flagged_sorted[:10]:
    exp = explainer.explain_node(idx)
    explainer.print_explanation(exp, is_fraud=(fraud_labels[idx] == 1))

# Explanation Precision@K
print("========")
print("EXPLANATION PRECISION EVALUATION (RQ4)")
print("========")

for k in [1, 2, 3]:
    r = evaluate_explanation_precision(explainer, fraud_labels, top_k=k)
    print(f"\n  Explanation Precision@{k}:")
    print(f"    Fraud tenders evaluated: {r['num_fraud']}")
    print(f"    Avg Precision@{k}:       {r['avg_precision_at_k']:.3f}")
    print(f"    Hit Rate (any correct):  {r['hit_rate']:.3f}")


Using model from comparison dict.

BUILDING EXPLAINABILITY MODULE (Objective 3)
  Fitting training residual distribution...
  Computing attention proxy scores...
  Computing per-feature z-scores...
  ✓ Explainability module ready

Flagged 2675 tenders above 95th percentile
Of which 2026 are known fraud (2026/2675 = 75.7%)

TOP 10 FLAGGED ENTITIES WITH EXPLANATIONS

  [  normal] Tender 45879  (score: 35.0784, percentile: 100.0)
    Relationships:
      Buyer 294 → issues (attn: 0.688)
    Deviations:
      log_tender_value (+7.3σ)
      procurement_method (+1.2σ)
      tender_status (-0.4σ)

  [  normal] Tender 39982  (score: 34.4241, percentile: 100.0)
    Deviations:
      log_tender_value (+7.1σ)
      tender_status (+1.5σ)
      procurement_method (-0.7σ)

  [  normal] Tender 17455  (score: 33.0961, percentile: 100.0)
    Relationships:
      Buyer 1312 → issues (attn: 0.463)
    Deviations:
      log_tender_value (+6.7σ)
      procurement_method (+1.9σ)
      tender_status (-0.5σ)


# Verification, Diagnostics & Scale Sweep

The cells below extend the 50,000 contracts run with reproducible multi-seed evaluation, a Tabular AE diagnostic, a dataset-size sweep, and ablations at scale.

**Run the cells in this section in order.** Each cell writes its results to `results/*.csv` and is safe to re-run — completed seeds are skipped on resume.

| Cell | Purpose |
|------|---------|
| Infrastructure | Seed control, memory cleanup, dataset builder, ablation variants |
| Diagnostic | Investigate Tabular AE AUC = 0.013 anomaly (run **first** — cheap) |
| Size sweep | (size × seed) grid over 1k / 5k / 10k / 20k / 50k records |
| Scale chart | Read sweep CSV, plot P@50 and AUC vs dataset size |
| Ablations: temporal + masking rate | Same model design used at small scales |
| Ablation: heterogeneity | TA-HGMAE on hetero vs homogenised graph (same architecture) × 3 seeds |
| Ablation: re-mask decoding | Standard vs GraphMAE-style re-mask decoding × 3 seeds |


In [ ]:
# ============================================================================
# REUSABLE INFRASTRUCTURE FOR SEEDED EXPERIMENTS
# ============================================================================
# Provides:
#   - set_all_seeds(seed): deterministic random / numpy / torch
#   - cleanup_memory():    free GPU memory between experiments
#   - parse_records_once(): cache parsed JSONL records across configs
#   - NoTemporalTA_HGMAE:  ablation variant (zeroed temporal context)
#
# Helpers for seeded experiment loops
# ============================================================================

import gc
import os
import random
import numpy as np
import pandas as pd
import torch
from datetime import datetime, timezone

def set_all_seeds(seed: int):
    """Make a run reproducible across random / numpy / torch / cuDNN."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup_memory():
    """Free GPU memory between experiments (important on Colab)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Module-level cache so we only parse the JSONL once per Colab session.
_RECORD_CACHE = {}

def parse_records_once(filepath: str, max_records: int):
    """Parse JSONL once and cache. Reused across all sweep configs."""
    key = (filepath, max_records)
    if key in _RECORD_CACHE:
        return _RECORD_CACHE[key]
    loader = OCDDataLoader(filepath)
    records = loader.load_data(max_records=max_records)
    _RECORD_CACHE[key] = records
    return records


def build_seeded_graph(records, num_records, num_burst_buyers, burst_size,
                       num_round, seed):
    """
    Build a hetero graph + fraud labels for a (size, seed) config.
    Holds the fraud rate at ~6.54% by scaling injection counts proportionally
    to the canonical 317090 / (500*5 + 1000) setup.
    """
    set_all_seeds(seed)
    sliced = records[:num_records]

    builder = ProcurementGraphBuilder()
    graph = builder.build_graph(sliced)

    injector = FraudInjector(graph)
    graph = injector.inject_burst_fraud(num_buyers=num_burst_buyers,
                                        burst_size=burst_size)
    graph = injector.inject_round_amounts(num_tenders=num_round)
    return graph, graph['Tender'].fraud_label


class NoTemporalTA_HGMAE(TA_HGMAE):
    def _aggregate_temporal_to_nodes(self, edge_temporal_features, data):
        return {
            ntype: torch.zeros(data[ntype].num_nodes, self.temporal_dim)
            for ntype in self.node_types
        }

In [ ]:
# DIAGNOSTIC: TABULAR AE AUC = 0.013 ANOMALY


from sklearn.metrics import roc_auc_score
import numpy as np

assert 'TabularAutoencoder' in dir(), "Run cell 5 first to define the baselines."
assert os.path.exists('data/processed/procurement_graph.pt'), \
    "Run cell 3 first to build the graph."

data = torch.load('data/processed/procurement_graph.pt', weights_only=False)
fraud_labels = data['Tender'].fraud_label

set_all_seeds(42)
in_dim = data['Tender'].x.shape[1]
ae = TabularAutoencoder(in_dim=in_dim, hidden_dim=32)
ae.fit(data, num_epochs=100, lr=0.001)

scores = ae.predict_scores(data).numpy()
labels = fraud_labels.numpy().astype(int)

print("\n" + "=========" )
print("TABULAR AE DIAGNOSTIC")
print(f"\nDataset: {len(labels)} tenders, {labels.sum()} fraud "
      f"({labels.mean():.2%})\n")

print("Score distribution (per-tender squared reconstruction error):")
print(f"  Overall:  mean={scores.mean():.5f}  std={scores.std():.5f}  "
      f"min={scores.min():.5f}  max={scores.max():.5f}")
print(f"  Legit:    mean={scores[labels==0].mean():.5f}  "
      f"std={scores[labels==0].std():.5f}  "
      f"median={np.median(scores[labels==0]):.5f}")
print(f"  Fraud:    mean={scores[labels==1].mean():.5f}  "
      f"std={scores[labels==1].std():.5f}  "
      f"median={np.median(scores[labels==1]):.5f}")

auc       = roc_auc_score(labels,  scores)
auc_inv   = roc_auc_score(labels, -scores)
print(f"\nAUC raw:      {auc:.4f}")
print(f"AUC inverted: {auc_inv:.4f}")

# Top / bottom 50 by score
order = np.argsort(scores)
top50 = order[-50:][::-1]
bot50 = order[:50]
top50_fraud = int(labels[top50].sum())
bot50_fraud = int(labels[bot50].sum())
print(f"\nTop-50    by score (highest error):  {top50_fraud}/50 fraud  "
      f"→ P@50 = {top50_fraud/50:.3f}")
print(f"Bottom-50 by score (lowest  error):  {bot50_fraud}/50 fraud")


# Also save for later reference
os.makedirs("results", exist_ok=True)
pd.DataFrame({
    'tender_idx':  np.arange(len(scores)),
    'score':       scores,
    'is_fraud':    labels,
}).to_csv("results/tabular_ae_diagnostic_scores.csv", index=False)
print("\nFull score-vs-label table saved: results/tabular_ae_diagnostic_scores.csv")


In [ ]:
# ============================================================================
# PRIORITY 1: THREE-SEED VERIFICATION AT 53,500 RECORDS
# ============================================================================
# Replicates the canonical 4-model comparison across seeds [42, 123, 7] and
# reports mean ± std. Each seed varies BOTH the fraud-injection randomness
# (which buyers, which values, which timestamps) AND the model initialisation.


import os
import pandas as pd

assert 'run_full_comparison' in dir(), "Run cell 5 first."

os.makedirs("results", exist_ok=True)
SEEDS = [42, 123, 7]
RESULTS_CSV = "results/three_seed_53500.csv"

# Cached records (single parse — reused across seeds)
records = parse_records_once('data/raw/colombia_procurement_sample.jsonl',
                             max_records=50000)

# Resume from CSV if a partial run exists
all_rows = []
done = set()
if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    all_rows = existing.to_dict('records')
    done = set(existing['seed'].astype(int).tolist())
    print(f"Found {RESULTS_CSV} — already completed seeds: {sorted(done)}")

for seed in SEEDS:
    if seed in done:
        print(f"\n[seed {seed}] SKIP (already in CSV).")
        continue

    print("\n" + "========")
    print(f"SEED {seed}")

    # Build graph + inject fraud (seeded)
    graph, fraud_labels = build_seeded_graph(
        records=records,
        num_records=50000,
        num_burst_buyers=500, burst_size=5, num_round=1000,
        seed=seed,
    )
    print(f"\n[seed {seed}] graph: {graph['Tender'].num_nodes} tenders  "
          f"→ {fraud_labels.sum().item()} fraud "
          f"({fraud_labels.float().mean().item():.2%})")

    # Re-seed before model training so model init / mask sampling are reproducible
    set_all_seeds(seed)
    _, results_df = run_full_comparison(graph, fraud_labels)

    results_df['seed'] = seed
    results_df['num_records']    = 50000
    results_df['total_tenders']  = graph['Tender'].num_nodes
    results_df['fraud_count']    = fraud_labels.sum().item()
    all_rows.extend(results_df.to_dict('records'))

    # Persist after every seed
    pd.DataFrame(all_rows).to_csv(RESULTS_CSV, index=False)
    print(f"\n[seed {seed}] saved to {RESULTS_CSV}")

    del graph, fraud_labels
    cleanup_memory()

# Aggregate across seeds
final = pd.DataFrame(all_rows)
print("\n" + "========")
print("THREE-SEED AGGREGATE — 53,500 RECORDS")
print("=========")

agg = final.groupby('Model').agg({
    'P@50':           ['mean', 'std'],
    'P@100':          ['mean', 'std'],
    'P@200':          ['mean', 'std'],
    'AUC-ROC':        ['mean', 'std'],
    'Detection Rate': ['mean', 'std'],
}).round(4)
print(agg.to_string())

agg.to_csv("results/three_seed_53500_aggregate.csv")
print("\nAggregate saved: results/three_seed_53500_aggregate.csv")
print(f"Per-seed rows:    {RESULTS_CSV}")
print(f"\nNote: IsolationForestBaseline uses random_state=42 internally, so its")
print(f"      seed-to-seed variation reflects only fraud-injection randomness,")
print(f"      not model variation.")


In [ ]:
# SIZE SWEEP: dataset_size × seed at fixed ~6.54% fraud rate
# Configs (real_records, burst_buyers, burst_size, round_amount_count):
#     1000  records → 70   fraud (10  × 5 burst + 20   round)  ≈ 6.54%
#     5000  records → 350  fraud (50  × 5 burst + 100  round)  ≈ 6.54%
#     10000 records → 700  fraud (100 × 5 burst + 200  round)  ≈ 6.54%
#     20000 records → 1400 fraud (200 × 5 burst + 400  round)  ≈ 6.54%
#     50000 records → 3500 fraud (500 × 5 burst + 1000 round)  ≈ 6.54%   (canonical)
#

import os
import pandas as pd

assert 'run_full_comparison' in dir(), "Run cell 5 first."

CONFIGS = [
    # (num_records, n_burst_buyers, burst_size, num_round)
    (1000,  10,  5, 20),
    (5000,  50,  5, 100),
    (10000, 100, 5, 200),
    (20000, 200, 5, 400),
    (50000, 500, 5, 1000),
]
SEEDS = [42, 123, 7]
SWEEP_CSV = "results/sweep_size_x_seed.csv"

records = parse_records_once('data/raw/colombia_procurement_sample.jsonl',
                             max_records=50000)

all_rows = []
done = set()
if os.path.exists(SWEEP_CSV):
    existing = pd.read_csv(SWEEP_CSV)
    all_rows = existing.to_dict('records')
    done = set(zip(existing['num_records'].astype(int).tolist(),
                   existing['seed'].astype(int).tolist()))

for (n_rec, n_buyers, burst, n_round) in CONFIGS:
    for seed in SEEDS:
        if (n_rec, seed) in done:
            print(f"[size={n_rec:>5d}  seed={seed}] SKIP (done).")
            continue

        print("\n" + "=======")
        print(f"SIZE = {n_rec}     SEED = {seed}")
        print("========")

        graph, fraud_labels = build_seeded_graph(
            records=records,
            num_records=n_rec,
            num_burst_buyers=n_buyers, burst_size=burst, num_round=n_round,
            seed=seed,
        )
        actual_rate = fraud_labels.float().mean().item()
        print(f"  total = {graph['Tender'].num_nodes}   "
              f"fraud = {fraud_labels.sum().item()}   "
              f"rate = {actual_rate:.2%}")

        set_all_seeds(seed)
        _, results_df = run_full_comparison(graph, fraud_labels)
        results_df['seed']           = seed
        results_df['num_records']    = n_rec
        results_df['total_tenders']  = graph['Tender'].num_nodes
        results_df['fraud_count']    = int(fraud_labels.sum().item())
        results_df['fraud_rate']     = actual_rate
        all_rows.extend(results_df.to_dict('records'))

        pd.DataFrame(all_rows).to_csv(SWEEP_CSV, index=False)
        print(f"  appended to {SWEEP_CSV}")

        del graph, fraud_labels
        cleanup_memory()

# Aggregate
sweep = pd.DataFrame(all_rows)
print("\n" + "=======" )
print("SIZE-SWEEP AGGREGATE (mean ± std across seeds)")
print("=======")

agg = sweep.groupby(['num_records', 'Model']).agg(
    p50_mean=('P@50',    'mean'),
    p50_std =('P@50',    'std'),
    p100_mean=('P@100',  'mean'),
    auc_mean=('AUC-ROC', 'mean'),
    auc_std =('AUC-ROC', 'std'),
).round(4)
print(agg.to_string())
agg.to_csv("results/sweep_aggregate.csv")
print("\nAggregate saved: results/sweep_aggregate.csv")


In [ ]:

# SCALE-DEPENDENCE CHART
# Reads the sweep CSV and plots P@50 and AUC vs dataset size for each model,
# with seed-level error bars. Saves PNG to results/scale_dependence.png so it
# can be embedded in Chapter 4 of the thesis.

import os
import pandas as pd
import matplotlib.pyplot as plt

assert os.path.exists("results/sweep_size_x_seed.csv"), \
    "Run the size-sweep cell first to produce results/sweep_size_x_seed.csv."

sweep = pd.read_csv("results/sweep_size_x_seed.csv")

agg = sweep.groupby(['num_records', 'Model']).agg(
    p50_mean=('P@50',    'mean'),
    p50_std =('P@50',    'std'),
    auc_mean=('AUC-ROC', 'mean'),
    auc_std =('AUC-ROC', 'std'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
preferred_order = ['TA-HGMAE', 'GraphMAE (Homo)', 'Tabular AE', 'Isolation Forest']
models = [m for m in preferred_order if m in agg['Model'].unique()] + \
         [m for m in agg['Model'].unique() if m not in preferred_order]
colors = {'TA-HGMAE': 'C0', 'GraphMAE (Homo)': 'C1',
          'Tabular AE': 'C2', 'Isolation Forest': 'C3'}
markers = {'TA-HGMAE': 'o', 'GraphMAE (Homo)': 's',
           'Tabular AE': '^', 'Isolation Forest': 'D'}

for m in models:
    sub = agg[agg['Model'] == m].sort_values('num_records')
    c = colors.get(m, None)
    mk = markers.get(m, 'o')
    axes[0].errorbar(sub['num_records'], sub['p50_mean'],
                     yerr=sub['p50_std'].fillna(0),
                     marker=mk, label=m, color=c, capsize=4, lw=2, ms=7)
    axes[1].errorbar(sub['num_records'], sub['auc_mean'],
                     yerr=sub['auc_std'].fillna(0),
                     marker=mk, label=m, color=c, capsize=4, lw=2, ms=7)

for ax, ylab, title in zip(axes,
                           ['Precision@50', 'AUC-ROC'],
                           ['P@50 vs Dataset Size', 'AUC-ROC vs Dataset Size']):
    ax.set_xscale('log')
    ax.set_xlabel('Number of real records (log scale)')
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=9, framealpha=0.9)

plt.tight_layout()
os.makedirs("results", exist_ok=True)
out_path = "results/scale_dependence.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")


In [ ]:
# ABLATION STUDIES — Temporal feature ablation (RQ2) and Masking rate sweep (RQ3)


import torch
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from sklearn.metrics import roc_auc_score
from copy import deepcopy
import os

os.makedirs('results', exist_ok=True)

# ---- Reuse evaluation primitives from the comparison cell ----
def precision_at_k_local(scores, labels, k=100):
    if k > len(scores):
        k = len(scores)
    top_k_idx = torch.topk(scores, k=k).indices
    return labels[top_k_idx].sum().item() / k

def evaluate_model(scores, fraud_labels):
    """Compute P@50, P@100, P@200, AUC, Detection Rate@100."""
    scores_np = scores.detach().numpy()
    labels_np = fraud_labels.numpy().astype(int)
    p50  = precision_at_k_local(scores, fraud_labels, 50)
    p100 = precision_at_k_local(scores, fraud_labels, 100)
    p200 = precision_at_k_local(scores, fraud_labels, 200)
    try:
        auc = roc_auc_score(labels_np, scores_np)
    except Exception:
        auc = 0.0
    top100 = torch.topk(scores, k=min(100, len(scores))).indices
    det = fraud_labels[top100].float().mean().item()
    return {'P@50': p50, 'P@100': p100, 'P@200': p200,
            'AUC-ROC': auc, 'Detection': det}

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

def train_and_score(model, data, current_time, num_epochs=100, lr=0.001):
    """Train then score in one go."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(num_epochs):
        optimizer.zero_grad()
        output = model(data, current_time)
        losses = model.compute_loss(output, data)
        losses['total'].backward()
        optimizer.step()
    scores = model.anomaly_score(data, current_time)['Tender']
    return scores

def disable_temporal(model):
    """
    Monkey-patch _aggregate_temporal_to_nodes to return zero tensors.
    Architecture and parameter count are unchanged; only the signal is killed.
    """
    temporal_dim = model.temporal_dim
    def _zero_aggregate(self, edge_temporal_features, data):
        return {
            ntype: torch.zeros(data[ntype].num_nodes, temporal_dim)
            for ntype in self.node_types
        }
    import types
    model._aggregate_temporal_to_nodes = types.MethodType(_zero_aggregate, model)
    return model


# ---- Load graph and fraud labels ----
data = torch.load('data/processed/procurement_graph.pt', weights_only=False)
fraud_labels = data['Tender'].fraud_label
current_time = datetime.now(timezone.utc)
node_types = list(data.node_types)
edge_types = list(data.edge_types)
node_feat_dims = {nt: data[nt].x.shape[1] for nt in node_types}

print("=======")
print("ABLATION STUDIES")
print("=======")
print(f"Dataset: {data['Tender'].num_nodes} tenders, "
      f"{fraud_labels.sum().item()} fraud, "
      f"{fraud_labels.float().mean().item():.2%} fraud rate")
print()

SEEDS = [42, 123, 7]

# ABLATION 1: TEMPORAL FEATURES (RQ2)
print("=======")
print("ABLATION 1: TEMPORAL FEATURES (RQ2)")
print("=======")

temporal_results = []

for variant_name, disable_temp in [
    ('TA-HGMAE (with temporal)', False),
    ('TA-HGMAE (no temporal)',   True),
]:
    print(f"\n  {variant_name}")
    print("  " + "-" * 60)
    for seed in SEEDS:
        set_seed(seed)
        model = TA_HGMAE(
            node_types=node_types, edge_types=edge_types,
            node_feat_dims=node_feat_dims,
            hidden_dim=32, num_layers=2, mask_rate=0.15, window_days=180,
        )
        # dummy forward pass to initialise lazy layers
        _ = model(data, current_time)
        if disable_temp:
            model = disable_temporal(model)
        scores = train_and_score(model, data, current_time, num_epochs=100)
        m = evaluate_model(scores, fraud_labels)
        m['variant'] = variant_name
        m['seed'] = seed
        temporal_results.append(m)
        print(f"    seed={seed}: P@50={m['P@50']:.3f}  P@100={m['P@100']:.3f}  "
              f"AUC={m['AUC-ROC']:.3f}  Det={m['Detection']:.1%}")

# Aggregate
df_t = pd.DataFrame(temporal_results)
agg_t = df_t.groupby('variant').agg(
    P50_mean=('P@50','mean'),    P50_std=('P@50','std'),
    P100_mean=('P@100','mean'),  P100_std=('P@100','std'),
    P200_mean=('P@200','mean'),  P200_std=('P@200','std'),
    AUC_mean=('AUC-ROC','mean'), AUC_std=('AUC-ROC','std'),
    Det_mean=('Detection','mean'),Det_std=('Detection','std'),
).round(3)

print("\n" + "=========")
print("TEMPORAL ABLATION SUMMARY (mean ± std over 3 seeds)")
print("=======")
print(agg_t.to_string())
df_t.to_csv('results/ablation_temporal.csv', index=False)
agg_t.to_csv('results/ablation_temporal_summary.csv')
print("\n  Saved: results/ablation_temporal.csv (per-seed)")
print("  Saved: results/ablation_temporal_summary.csv (aggregated)")


# ABLATION 2: MASKING RATE SWEEP (RQ3)

print("\n" + "========")
print("ABLATION 2: MASKING RATE SWEEP (RQ3)")
print("=======")

mask_results = []

for mu in [0.15, 0.40, 0.75]:
    print(f"\n  TA-HGMAE (mask_rate = {mu})")
    print("  " + "-" * 60)
    for seed in SEEDS:
        set_seed(seed)
        model = TA_HGMAE(
            node_types=node_types, edge_types=edge_types,
            node_feat_dims=node_feat_dims,
            hidden_dim=32, num_layers=2, mask_rate=mu, window_days=180,
        )
        _ = model(data, current_time)
        scores = train_and_score(model, data, current_time, num_epochs=100)
        m = evaluate_model(scores, fraud_labels)
        m['mask_rate'] = mu
        m['seed'] = seed
        mask_results.append(m)
        print(f"    seed={seed}: P@50={m['P@50']:.3f}  P@100={m['P@100']:.3f}  "
              f"AUC={m['AUC-ROC']:.3f}  Det={m['Detection']:.1%}")

df_m = pd.DataFrame(mask_results)
agg_m = df_m.groupby('mask_rate').agg(
    P50_mean=('P@50','mean'),    P50_std=('P@50','std'),
    P100_mean=('P@100','mean'),  P100_std=('P@100','std'),
    P200_mean=('P@200','mean'),  P200_std=('P@200','std'),
    AUC_mean=('AUC-ROC','mean'), AUC_std=('AUC-ROC','std'),
    Det_mean=('Detection','mean'),Det_std=('Detection','std'),
).round(3)

print("\n" + "======")
print("MASKING RATE SUMMARY (mean ± std over 3 seeds)")
print("========")
print(agg_m.to_string())
df_m.to_csv('results/ablation_mask_rate.csv', index=False)
agg_m.to_csv('results/ablation_mask_rate_summary.csv')
print("\n  Saved: results/ablation_mask_rate.csv (per-seed)")
print("  Saved: results/ablation_mask_rate_summary.csv (aggregated)")

print("=======")
with_t  = df_t[df_t['variant'] == 'TA-HGMAE (with temporal)']['P@50'].mean()
no_t    = df_t[df_t['variant'] == 'TA-HGMAE (no temporal)']['P@50'].mean()
delta   = with_t - no_t
print(f"\nTemporal contribution (P@50): {with_t:.3f} vs {no_t:.3f} "
      f"(Δ = {delta:+.3f})")
best_mu = agg_m['P50_mean'].idxmax()
print(f"\nBest masking rate by P@50: μ = {best_mu} "
      f"(P@50 = {agg_m.loc[best_mu, 'P50_mean']:.3f})")




ABLATION STUDIES
Dataset: 53500 tenders, 3500 fraud, 6.54% fraud rate

ABLATION 1: TEMPORAL FEATURES (RQ2)

  TA-HGMAE (with temporal)
  ------------------------------------------------------------
    seed=42: P@50=0.040  P@100=0.230  AUC=0.968  Det=23.0%
    seed=123: P@50=0.080  P@100=0.450  AUC=0.982  Det=45.0%
    seed=7: P@50=0.160  P@100=0.520  AUC=0.979  Det=52.0%

  TA-HGMAE (no temporal)
  ------------------------------------------------------------
    seed=42: P@50=0.000  P@100=0.360  AUC=0.945  Det=36.0%
    seed=123: P@50=0.000  P@100=0.350  AUC=0.971  Det=35.0%
    seed=7: P@50=0.000  P@100=0.380  AUC=0.965  Det=38.0%

TEMPORAL ABLATION SUMMARY (mean ± std over 3 seeds)
                          P50_mean  P50_std  P100_mean  P100_std  P200_mean  P200_std  AUC_mean  AUC_std  Det_mean  Det_std
variant                                                                                                                    
TA-HGMAE (no temporal)       0.000    0.000      0.363    

In [ ]:
# ============================================================================
# ABLATION 3: HETEROGENEITY (extends RQ1)
# Compares TA-HGMAE on the heterogeneous Buyer↔Tender graph against the same
# architecture trained on a homogenised version where all nodes are pooled
# into a single type. This isolates the value of type-dependent attention
# independent of message passing or self-supervision.
#
# Approach: build a homogenised hetero graph (all nodes promoted to type
# "Node", padded to common feature dim, edges relabelled "connects").
# Train the same TA_HGMAE class on it. Compare metrics at 3 seeds.

import torch
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from sklearn.metrics import roc_auc_score
from torch_geometric.data import HeteroData
from copy import deepcopy
import os

os.makedirs('results', exist_ok=True)

# ---- Reuse evaluation primitives from the previous ablation cell ----
# (precision_at_k_local, evaluate_model, set_seed, train_and_score should
#  already be in scope. If not, re-define them — they're at the top of the
#  previous ablation cell.)


def homogenise_hetero(data):
    """
    Build a homogenised HeteroData object where all nodes share a single
    type "Node" and all edges share a single type ("Node", "connects", "Node").

    All node features are padded to the maximum feature dimension across
    original types, then concatenated. Edge attributes (timestamps) are
    preserved on the new single edge type. Fraud labels are remapped to
    the corresponding nodes in the new "Node" type.
    """
    # 1. Determine padded feature dim
    max_dim = max(data[nt].x.shape[1] for nt in data.node_types)

    # 2. Build offset map and concatenated node features
    offsets = {}
    cursor = 0
    padded_xs = []
    fraud_labels_list = []

    for nt in data.node_types:
        x = data[nt].x
        n = x.shape[0]
        offsets[nt] = cursor
        # Pad features to max_dim
        if x.shape[1] < max_dim:
            pad = torch.zeros(n, max_dim - x.shape[1])
            x = torch.cat([x, pad], dim=1)
        padded_xs.append(x)

        # Carry over fraud labels for Tender; zero-pad for other types
        if hasattr(data[nt], 'fraud_label') and data[nt].fraud_label is not None:
            fraud_labels_list.append(data[nt].fraud_label)
        else:
            fraud_labels_list.append(torch.zeros(n, dtype=torch.long))

        cursor += n

    all_x = torch.cat(padded_xs, dim=0)
    all_fraud = torch.cat(fraud_labels_list, dim=0)

    # 3. Concatenate all edges, remap indices through offsets
    all_src, all_dst, all_attr = [], [], []
    for et in data.edge_types:
        src_t, _, dst_t = et
        ei = data[et].edge_index
        all_src.append(ei[0] + offsets[src_t])
        all_dst.append(ei[1] + offsets[dst_t])
        if hasattr(data[et], 'edge_attr') and data[et].edge_attr is not None:
            all_attr.append(data[et].edge_attr)

    edge_index = torch.stack([torch.cat(all_src), torch.cat(all_dst)], dim=0)
    edge_attr = torch.cat(all_attr, dim=0) if all_attr else None

    # 4. Build a new HeteroData with one node type and one edge type
    homo = HeteroData()
    homo['Node'].x = all_x
    homo['Node'].num_nodes = all_x.shape[0]
    homo['Node'].fraud_label = all_fraud
    homo[('Node', 'connects', 'Node')].edge_index = edge_index
    if edge_attr is not None:
        homo[('Node', 'connects', 'Node')].edge_attr = edge_attr

    # store the offset for Tender so we can slice scores back later
    homo._tender_offset = offsets['Tender']
    homo._tender_count = data['Tender'].num_nodes

    return homo


# ---- Load data and prepare both versions ----
data_hetero = torch.load('data/processed/procurement_graph.pt', weights_only=False)
fraud_labels_full = data_hetero['Tender'].fraud_label
current_time = datetime.now(timezone.utc)

data_homo = homogenise_hetero(data_hetero)

print(f"Heterogeneous: Buyer={data_hetero['Buyer'].num_nodes}, "
      f"Tender={data_hetero['Tender'].num_nodes}, "
      f"edges={data_hetero[('Buyer','issues','Tender')].edge_index.shape[1]}")
print(f"Homogenised:   Node={data_homo['Node'].num_nodes}, "
      f"edges={data_homo[('Node','connects','Node')].edge_index.shape[1]}")
print(f"Tender slice in homo: offset={data_homo._tender_offset}, "
      f"count={data_homo._tender_count}\n")


def evaluate_homo(model, data_homo, fraud_labels):
    """Score all nodes, then slice out the Tender slab to compare like-for-like."""
    scores_all = model.anomaly_score(data_homo, current_time)['Node']
    off = data_homo._tender_offset
    n = data_homo._tender_count
    tender_scores = scores_all[off:off + n]
    return evaluate_model(tender_scores, fraud_labels)


# RUN

print("=========")
print("ABLATION 3: HETEROGENEITY (extends RQ1)")
print("=========")

results = []
SEEDS = [42, 123, 7]

# --- Heterogeneous variant (Buyer + Tender + 'issues' edges) ---
print("\n  TA-HGMAE (heterogeneous: Buyer + Tender)")
print("  " + "-----------")
for seed in SEEDS:
    set_seed(seed)
    node_types = list(data_hetero.node_types)
    edge_types = list(data_hetero.edge_types)
    node_feat_dims = {nt: data_hetero[nt].x.shape[1] for nt in node_types}
    model = TA_HGMAE(
        node_types=node_types, edge_types=edge_types,
        node_feat_dims=node_feat_dims,
        hidden_dim=32, num_layers=2, mask_rate=0.15, window_days=180,
    )
    _ = model(data_hetero, current_time)  # init lazy layers
    scores = train_and_score(model, data_hetero, current_time, num_epochs=100)
    m = evaluate_model(scores, fraud_labels_full)
    m['variant'] = 'TA-HGMAE (heterogeneous)'
    m['seed'] = seed
    results.append(m)
    print(f"    seed={seed}: P@50={m['P@50']:.3f}  P@100={m['P@100']:.3f}  "
          f"AUC={m['AUC-ROC']:.3f}  Det={m['Detection']:.1%}")

# --- Homogenised variant (single 'Node' type) ---
print("\n  TA-HGMAE (homogenised: single Node type)")
print("  " + "---------")
for seed in SEEDS:
    set_seed(seed)
    node_types_h = list(data_homo.node_types)
    edge_types_h = list(data_homo.edge_types)
    node_feat_dims_h = {nt: data_homo[nt].x.shape[1] for nt in node_types_h}
    model = TA_HGMAE(
        node_types=node_types_h, edge_types=edge_types_h,
        node_feat_dims=node_feat_dims_h,
        hidden_dim=32, num_layers=2, mask_rate=0.15, window_days=180,
    )
    _ = model(data_homo, current_time)
    # Train normally, then evaluate only on the Tender slab
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for _ in range(100):
        optimizer.zero_grad()
        output = model(data_homo, current_time)
        losses = model.compute_loss(output, data_homo)
        losses['total'].backward()
        optimizer.step()
    m = evaluate_homo(model, data_homo, fraud_labels_full)
    m['variant'] = 'TA-HGMAE (homogenised)'
    m['seed'] = seed
    results.append(m)
    print(f"    seed={seed}: P@50={m['P@50']:.3f}  P@100={m['P@100']:.3f}  "
          f"AUC={m['AUC-ROC']:.3f}  Det={m['Detection']:.1%}")


# ---- Aggregate ----
df = pd.DataFrame(results)
agg = df.groupby('variant').agg(
    P50_mean=('P@50','mean'),    P50_std=('P@50','std'),
    P100_mean=('P@100','mean'),  P100_std=('P@100','std'),
    P200_mean=('P@200','mean'),  P200_std=('P@200','std'),
    AUC_mean=('AUC-ROC','mean'), AUC_std=('AUC-ROC','std'),
    Det_mean=('Detection','mean'),Det_std=('Detection','std'),
).round(3)

print("\n" + "========")
print("HETEROGENEITY ABLATION SUMMARY (mean ± std over 3 seeds)")
print("=======")
print(agg.to_string())
df.to_csv('results/ablation_heterogeneity.csv', index=False)
agg.to_csv('results/ablation_heterogeneity_summary.csv')
print("\n  Saved: results/ablation_heterogeneity.csv (per-seed)")
print("  Saved: results/ablation_heterogeneity_summary.csv (aggregated)")


# ---- Quick interpretation hint ----
print("\n" + "========")
print("QUICK INTERPRETATION")
print("========")
het = df[df['variant'] == 'TA-HGMAE (heterogeneous)']
hom = df[df['variant'] == 'TA-HGMAE (homogenised)']
delta_p50 = het['P@50'].mean() - hom['P@50'].mean()
delta_auc = het['AUC-ROC'].mean() - hom['AUC-ROC'].mean()
print(f"\nP@50:    hetero {het['P@50'].mean():.3f} - homo {hom['P@50'].mean():.3f} "
      f"= Δ {delta_p50:+.3f}")
print(f"AUC-ROC: hetero {het['AUC-ROC'].mean():.3f} - homo {hom['AUC-ROC'].mean():.3f} "
      f"= Δ {delta_auc:+.3f}")


Heterogeneous: Buyer=1877, Tender=53500, edges=50760
Homogenised:   Node=55377, edges=50760
Tender slice in homo: offset=1877, count=53500

ABLATION 3: HETEROGENEITY (extends RQ1)

  TA-HGMAE (heterogeneous: Buyer + Tender)
  ------------------------------------------------------------
    seed=42: P@50=0.080  P@100=0.260  AUC=0.958  Det=26.0%
    seed=123: P@50=0.020  P@100=0.390  AUC=0.970  Det=39.0%
    seed=7: P@50=0.220  P@100=0.500  AUC=0.978  Det=50.0%

  TA-HGMAE (homogenised: single Node type)
  ------------------------------------------------------------
    seed=42: P@50=0.000  P@100=0.370  AUC=0.980  Det=37.0%
    seed=123: P@50=0.120  P@100=0.390  AUC=0.978  Det=39.0%
    seed=7: P@50=0.080  P@100=0.450  AUC=0.976  Det=45.0%

HETEROGENEITY ABLATION SUMMARY (mean ± std over 3 seeds)
                          P50_mean  P50_std  P100_mean  P100_std  P200_mean  P200_std  AUC_mean  AUC_std  Det_mean  Det_std
variant                                                               

In [ ]:
# Priority 1 follow-up: Tabular AE held-out control experiment
# Tests the reconstruction-trap hypothesis
# vanilla AE ONLY on un-injected (legit) tenders, then scoring the full
# 53,500-tender set as usual. Architecture and hyperparameters are identical
# to the contaminated run in cell 10 -- the ONLY thing changing is whether
# the synthetic fraud tenders are present in the training data.
# Resume support like cell 10 -- saves to CSV after each seed.

import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

assert 'TabularAutoencoder' in dir(), "Run cell 5 first to define the baselines."
assert 'build_seeded_graph' in dir(), "Run cell 8 first for the seeded-graph helper."
assert 'precision_at_k' in dir(),    "Run cell 5 first for precision_at_k."

os.makedirs("results", exist_ok=True)
SEEDS = [42, 123, 7]
RESULTS_CSV = "results/tabular_ae_heldout_53500.csv"

records = parse_records_once('data/raw/colombia_procurement_sample.jsonl',
                             max_records=50000)

all_rows = []
done = set()
if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    all_rows = existing.to_dict('records')
    done = set(existing['seed'].astype(int).tolist())
    print(f"Found {RESULTS_CSV} -- already completed seeds: {sorted(done)}")

for seed in SEEDS:
    if seed in done:
        print(f"\n[seed {seed}] SKIP (already in CSV).")
        continue

    print("\n" + "=============")
    print(f"SEED {seed}  (Tabular AE held-out)")
    print("=======")

    graph, fraud_labels = build_seeded_graph(
        records=records,
        num_records=50000,
        num_burst_buyers=500, burst_size=5, num_round=1000,
        seed=seed,
    )
    X = graph['Tender'].x
    y = fraud_labels

    clean_mask = (y == 0)
    X_clean = X[clean_mask]
    print(f"\n[seed {seed}] full set:        {X.shape[0]:,} tenders "
          f"({y.sum().item()} fraud, {(y == 0).sum().item()} legit)")
    print(f"[seed {seed}] held-out train:  {X_clean.shape[0]:,} legit tenders")
    print(f"[seed {seed}] scored set:      {X.shape[0]:,} tenders (full)")

    # Re-seed before model init so model initialisation matches cell 10's
    # per-seed protocol (set_all_seeds before each model build).
    set_all_seeds(seed)
    ae = TabularAutoencoder(in_dim=X.shape[1], hidden_dim=32)

    # SAME training recipe as TabularAutoencoder.fit() -- only the data changes.
    optimizer = torch.optim.Adam(ae.parameters(), lr=0.001)
    print(f"\n[seed {seed}] training AE on clean subset (100 epochs, lr=1e-3)...")
    for epoch in range(100):
        ae.train()
        optimizer.zero_grad()
        loss = F.mse_loss(ae(X_clean), X_clean)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d}/100, Loss: {loss.item():.6f}")

    # Score the FULL 53,500-tender set
    ae.eval()
    with torch.no_grad():
        x_recon = ae(X)
        scores = ((X - x_recon) ** 2).mean(dim=1)

    scores_np = scores.numpy()
    labels_np = y.numpy().astype(int)

    auc  = roc_auc_score(labels_np, scores_np)
    p50  = precision_at_k(scores, y, k=50)
    p100 = precision_at_k(scores, y, k=100)
    p200 = precision_at_k(scores, y, k=200)
    top100_idx = torch.topk(scores, k=min(100, len(scores))).indices
    det100 = y[top100_idx].float().mean().item()

    legit_err = float(scores_np[labels_np == 0].mean())
    fraud_err = float(scores_np[labels_np == 1].mean())
    ratio     = fraud_err / legit_err if legit_err > 0 else float('inf')

    print(f"\n[seed {seed}] AUC-ROC:        {auc:.4f}")
    print(f"[seed {seed}] P@50 / P@100:   {p50:.3f} / {p100:.3f}")
    print(f"[seed {seed}] legit recon err: {legit_err:.5f}")
    print(f"[seed {seed}] fraud recon err: {fraud_err:.5f}")
    print(f"[seed {seed}] ratio f/l:      {ratio:.3f}",
          "  -- trap RESOLVED" if ratio > 1 else "  -- trap persists")

    all_rows.append({
        'seed': seed,
        'P@50': p50, 'P@100': p100, 'P@200': p200,
        'AUC-ROC': auc, 'Detection Rate': det100,
        'legit_recon_err': legit_err,
        'fraud_recon_err': fraud_err,
        'fraud_legit_ratio': ratio,
        'num_records': 50000,
        'total_tenders': X.shape[0],
        'fraud_count': int(y.sum().item()),
    })

    pd.DataFrame(all_rows).to_csv(RESULTS_CSV, index=False)
    print(f"[seed {seed}] saved to {RESULTS_CSV}")

    del graph, fraud_labels, X, y, X_clean, ae
    cleanup_memory()

# Aggregate
final = pd.DataFrame(all_rows).sort_values('seed').reset_index(drop=True)
print("\n" + "=============")
print("TABULAR AE HELD-OUT -- 53,500 RECORDS, 3 SEEDS")
print("=======")
print(final[['seed', 'AUC-ROC', 'P@50', 'P@100',
             'legit_recon_err', 'fraud_recon_err',
             'fraud_legit_ratio']].to_string(index=False))

print("\nAggregate (mean +/- std across seeds):")
print(f"  AUC-ROC:     {final['AUC-ROC'].mean():.3f} +/- {final['AUC-ROC'].std():.3f}")
print(f"  P@50:        {final['P@50'].mean():.3f} +/- {final['P@50'].std():.3f}")
print(f"  P@100:       {final['P@100'].mean():.3f} +/- {final['P@100'].std():.3f}")
print(f"  legit err:   {final['legit_recon_err'].mean():.5f}")
print(f"  fraud err:   {final['fraud_recon_err'].mean():.5f}")
print(f"  ratio f/l:   {final['fraud_legit_ratio'].mean():.3f}")

print("-----------")
print(f"{'metric':<22}{'contaminated (paper)':<24}{'held-out (this run)':<24}")
print(f"{'AUC mean +/- std':<22}{'0.338 +/- 0.561':<24}"
      f"{final['AUC-ROC'].mean():.3f} +/- {final['AUC-ROC'].std():.3f}")
print(f"{'legit recon err':<22}{'0.02087':<24}{final['legit_recon_err'].mean():.5f}")
print(f"{'fraud recon err':<22}{'0.00448':<24}{final['fraud_recon_err'].mean():.5f}")
print(f"{'ratio (fraud/legit)':<22}{'0.215':<24}{final['fraud_legit_ratio'].mean():.3f}")

print("\n" + "--------")
all_aucs_proper  = (final['AUC-ROC'] > 0.5).all()
all_ratios_flip  = (final['fraud_legit_ratio'] > 1).all()
final.to_csv(RESULTS_CSV, index=False)
print(f"\nPer-seed CSV: {RESULTS_CSV}")

In [ ]:
# Priority 4 ablation: GraphMAE-style re-mask decoding vs. standard decoding.
# Same architecture, same data, same seeds -- only the boolean use_remask flips.
# Helps to isolate whether re-masking adds value on the heterogeneous schema.

import torch
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from sklearn.metrics import roc_auc_score
import os

os.makedirs('results', exist_ok=True)


assert 'evaluate_model' in dir(),    "Run cell 13 first to define evaluate_model."
assert 'train_and_score' in dir(),   "Run cell 13 first to define train_and_score."
assert 'set_seed' in dir(),          "Run cell 13 first to define set_seed."

# ---- Load graph and fraud labels (same as cell 13) ----
data = torch.load('data/processed/procurement_graph.pt', weights_only=False)
fraud_labels = data['Tender'].fraud_label
current_time = datetime.now(timezone.utc)
node_types = list(data.node_types)
edge_types = list(data.edge_types)
node_feat_dims = {nt: data[nt].x.shape[1] for nt in node_types}

print("=======")
print("ABLATION 3: RE-MASK DECODING (Priority 4)")
print("=======")
print(f"Dataset: {data['Tender'].num_nodes} tenders, "
      f"{fraud_labels.sum().item()} fraud, "
      f"{fraud_labels.float().mean().item():.2%} fraud rate\n")

SEEDS = [42, 123, 7]
remask_results = []

for variant_name, use_remask_flag in [
    ('TA-HGMAE (standard decoding)',  False),
    ('TA-HGMAE (re-mask decoding)',   True),
]:
    print(f"\n  {variant_name}")
    print("  " + "-" * 60)
    for seed in SEEDS:
        set_seed(seed)
        model = TA_HGMAE(
            node_types=node_types, edge_types=edge_types,
            node_feat_dims=node_feat_dims,
            hidden_dim=32, num_layers=2, mask_rate=0.15, window_days=180,
            use_remask=use_remask_flag,
        )
        _ = model(data, current_time)         # dummy fwd to init lazy layers
        scores = train_and_score(model, data, current_time, num_epochs=100)
        m = evaluate_model(scores, fraud_labels)
        m['variant'] = variant_name
        m['seed']    = seed
        m['use_remask'] = use_remask_flag
        remask_results.append(m)
        print(f"    seed={seed}: P@50={m['P@50']:.3f}  P@100={m['P@100']:.3f}  "
              f"P@200={m['P@200']:.3f}  AUC={m['AUC-ROC']:.3f}  "
              f"Det={m['Detection']:.1%}")

# Aggregate
df_r = pd.DataFrame(remask_results)
agg_r = df_r.groupby('variant').agg(
    P50_mean=('P@50','mean'),    P50_std=('P@50','std'),
    P100_mean=('P@100','mean'),  P100_std=('P@100','std'),
    P200_mean=('P@200','mean'),  P200_std=('P@200','std'),
    AUC_mean=('AUC-ROC','mean'), AUC_std=('AUC-ROC','std'),
    Det_mean=('Detection','mean'),Det_std=('Detection','std'),
).round(3)

print("\n" + "=" * 70)
print("RE-MASK DECODING ABLATION SUMMARY (mean +/- std over 3 seeds)")
print("=======")
print(agg_r.to_string())

df_r.to_csv('results/ablation_remask.csv', index=False)
agg_r.to_csv('results/ablation_remask_summary.csv')
print("\n  Saved: results/ablation_remask.csv (per-seed)")
print("  Saved: results/ablation_remask_summary.csv (aggregated)")

# Quick interpretation
print("\n" + "=" * 70)
print("QUICK INTERPRETATION (use this to decide §4.3.4 framing)")
print("=======")

std_p50    = df_r[df_r['use_remask'] == False]['P@50'].mean()
remask_p50 = df_r[df_r['use_remask'] == True]['P@50'].mean()
delta_p50  = remask_p50 - std_p50

std_auc    = df_r[df_r['use_remask'] == False]['AUC-ROC'].mean()
remask_auc = df_r[df_r['use_remask'] == True]['AUC-ROC'].mean()
delta_auc  = remask_auc - std_auc

print(f"\n  P@50:   standard={std_p50:.3f}  re-mask={remask_p50:.3f}  "
      f"delta={delta_p50:+.3f}")
print(f"  AUC:    standard={std_auc:.3f}  re-mask={remask_auc:.3f}  "
      f"delta={delta_auc:+.3f}")
